# Viewpoint Estimation via kNN Graph Laplacian

Build a kNN graph on Fisher Vectors, use Laplacian eigenvectors to recover
a continuous viewing angle. Handle partial visibility via regression.

## Key Results
- **Laplacian 4D (raw)**: AUC 0.82, 74%/80% strict/relaxed
- **Laplacian 4D (vis-cleaned)**: AUC 0.85, 79%/84% strict/relaxed
- **EM masked centroid**: 84%/89% classification (handles partial visibility)
- EV1 strongly correlated with visibility (r=-0.84)
- Regressing out visibility improves metric by +0.03 AUC, +5% accuracy
- Remaining errors: low-visibility detections with genuinely ambiguous features


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import pickle
import time
from pathlib import Path
from collections import Counter, defaultdict
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
from scipy.spatial.distance import pdist, squareform

from src.config.config import MainConfig
from src.data.preprocessed_dataset import PreprocessedDataset
from src.data.annotation_loader import load_annotations
from src.evaluation import (
    load_or_compute_matching, COARSE_MAP, coarse_viewpoint,
    relaxed_correct, relaxed_same_viewpoint, pairwise_same_class_auc,
)
from src.codebook.gmm_trainer import load_gmm_model
from src.features.fisher_vector import build_block_mask, normalize_fvs
from src.laplacian import (
    build_knn_graph, normalized_laplacian, smallest_eigenvectors,
    angles_from_evs, angle_pair, ev_health_check, find_bottleneck_evs,
)
from src.utils.circular_stats import circular_mean, circular_std, circular_distance, circular_midpoint
from src.visualization.viewpoint_colors import VP_COLORS_8, VP_COLORS_4

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'


In [ ]:
# Load config, dataset, raw FVs, matching
config = MainConfig.from_yaml(Path('../config_seals.yaml'))
dataset = PreprocessedDataset(config.output_root)
coco_loader = load_annotations(config)

# Raw weight Fisher vectors (K=64)
raw_pkl_path = config.output_root / 'weight_fisher_vectors_raw.pkl'
print(f'Loading raw weight FVs from {raw_pkl_path}')
with open(raw_pkl_path, 'rb') as f:
    raw_data = pickle.load(f)
all_det_ids = raw_data['det_ids']
all_fvs_raw = raw_data['fvs_raw']
del raw_data

# GMM
gmm, _ = load_gmm_model(config.gmm_model_path)
K = gmm.n_components
D = gmm.means_.shape[1]
block = 2 * D + 1
print(f'GMM: K={K}, D={D}, FV dim={K * block}')
print(f'Raw FVs: {len(all_det_ids)} detections, shape={all_fvs_raw.shape}')

# GT matching
matched = load_or_compute_matching(
    dataset, coco_loader, config.output_root,
    target_size=config.active_resize_size,
    patch_size=config.active_patch_size,
    category_names=config.matching_categories,
    min_overlap_fraction=0.5,
)
print(f'Matched: {len(matched)} detections')

det_to_viewpoint = {}
det_to_image = {}
for m in matched:
    det_id = m.detection_id
    ann = m.gt_annotation
    det_to_viewpoint[det_id] = ann.viewpoint
    det_to_image[det_id] = ann.image_uuid

det_id_to_idx = {d: i for i, d in enumerate(all_det_ids)}

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

# FV components to use
USE_WEIGHT = True
USE_MEAN   = True
USE_VAR    = True

# Truncate mean/var to first N PCA dims (None = all 256)
MAX_PCA_DIM = None

# kNN graph
KNN_K = 10

# Coarse viewpoints (right/left/front/back) vs fine-grained
USE_COARSE_VP = True

# GMM granularity: None = use saved GMM; int = refit
N_GMM_COMPONENTS = None

# Prototype construction
N_EXAMPLES = 10            # examples per viewpoint for prototype
AGGREGATE_MODE = 'patches'   # 'mean' = average normalized FVs; 'patches' = FV from pooled patches
RANDOM_SEED = 42

# ============================================================
# Refit GMM if needed
# ============================================================
if N_GMM_COMPONENTS is not None and N_GMM_COMPONENTS != K:
    _fv_cache_path = config.output_root / f'weight_fisher_vectors_raw_K{N_GMM_COMPONENTS}.pkl'
    _gmm_cache_path = config.output_root / 'models' / f'gmm_K{N_GMM_COMPONENTS}.pkl'

    if _fv_cache_path.exists():
        print(f'Loading cached FVs from {_fv_cache_path}')
        with open(_fv_cache_path, 'rb') as f:
            _cached = pickle.load(f)
        all_fvs_raw = _cached['fvs_raw']
        K = N_GMM_COMPONENTS
        D = (all_fvs_raw.shape[1] // K - 1) // 2
        block = 2 * D + 1
        del _cached
        print(f'  Loaded {all_fvs_raw.shape[0]} FVs, K={K}, D={D}, dim={K*block}')
    else:
        raise FileNotFoundError(
            f'No cached FVs for K={N_GMM_COMPONENTS}. '
            f'Run miewid_vs_dino_fv.ipynb with N_GMM_COMPONENTS={N_GMM_COMPONENTS} first.'
        )

    if _gmm_cache_path.exists():
        with open(_gmm_cache_path, 'rb') as f:
            gmm = pickle.load(f)
        print(f'Loaded cached GMM K={gmm.n_components}')
    elif AGGREGATE_MODE == 'patches':
        raise FileNotFoundError(f'No cached GMM at {_gmm_cache_path}')

# ============================================================
# Build FV mask and normalize (using src.features.fisher_vector)
# ============================================================
fv_mask = build_block_mask(K, D, USE_WEIGHT, USE_MEAN, USE_VAR, MAX_PCA_DIM)
all_fvs_norm = normalize_fvs(all_fvs_raw, fv_mask)

active = []
if USE_WEIGHT: active.append('weight')
if USE_MEAN: active.append('mean')
if USE_VAR: active.append('var')
print(f'FV components: {"+".join(active)}, mask: {fv_mask.sum()}/{len(fv_mask)} active dims')

# ============================================================
# Coarse viewpoint mapping (using src.evaluation.viewpoint_accuracy)
# ============================================================
_COARSE_MAP = COARSE_MAP  # local alias for backward compat
if USE_COARSE_VP:
    VP_MAP = _COARSE_MAP
else:
    VP_MAP = {v: v for v in _COARSE_MAP.keys()}
    VP_MAP.update({'front': 'front', 'back': 'back'})

# Filter to detections with valid viewpoints
valid_dets = []
valid_vps = []
valid_raw_vps = []
for det_id in all_det_ids:
    vp = det_to_viewpoint.get(det_id, 'unknown')
    if vp in ('ignore', 'unknown'):
        continue
    coarse = VP_MAP.get(vp)
    if coarse is None:
        continue
    valid_dets.append(det_id)
    valid_vps.append(coarse)
    valid_raw_vps.append(vp)

valid_fv_idx = np.array([det_id_to_idx[d] for d in valid_dets])
valid_fvs = all_fvs_norm[valid_fv_idx]
valid_vps = np.array(valid_vps)
valid_raw_vps = np.array(valid_raw_vps)
del all_fvs_norm

vp_counts = Counter(valid_vps)
print(f'\nValid detections: {len(valid_dets)}')
for vp, cnt in sorted(vp_counts.items(), key=lambda x: -x[1]):
    print(f'  {vp:>15}: {cnt:5d}')

unique_vps = sorted(set(valid_vps))
PURE_VPS = set(unique_vps)
is_pure_vp = np.array([rv in PURE_VPS for rv in valid_raw_vps])
print(f'Unique viewpoints: {unique_vps}')
print(f'Pure VP detections: {is_pure_vp.sum()}/{len(is_pure_vp)} ({is_pure_vp.mean():.1%})')


In [ ]:
# ============================================================
# Angle recovery from graph Laplacian eigenvectors
# ============================================================
# If the data lies approximately on a circle (viewpoint orbit),
# the first two nontrivial Laplacian eigenvectors should trace
# a circle, and atan2(ev2, ev1) gives the angle.

# Primary EV pair for angle-based analyses (0-based index; EV0 is constant)
EV_X, EV_Y = 3, 4
EV_X_NAME, EV_Y_NAME = f'EV{EV_X}', f'EV{EV_Y}'
print(f'Using primary EV pair: {EV_X_NAME}, {EV_Y_NAME}')

# Build kNN graph + Laplacian using src.laplacian
print(f'Building {KNN_K}-NN graph on {valid_fvs.shape[0]} FVs...')
t0_knn = time.time()
A, degrees = build_knn_graph(valid_fvs, k=KNN_K)
L_sym = normalized_laplacian(A, degrees)
print(f'  Graph + Laplacian built in {time.time()-t0_knn:.1f}s')

print('Computing normalized Laplacian eigenvectors...')
eigenvalues_lap, eigenvectors_lap = smallest_eigenvectors(L_sym, n_eig=6)
print(f'  Laplacian eigenvalues: {eigenvalues_lap}')

ev_lap1 = eigenvectors_lap[:, EV_X]  # configurable primary EV X (EV0 is constant)
ev_lap2 = eigenvectors_lap[:, EV_Y]
theta_lap = np.arctan2(ev_lap2, ev_lap1)

# --- Visualize ---
vp_colors_map = {'right': 'blue', 'left': 'red', 'front': 'green', 'back': 'orange'}
vp_order = ['right', 'left', 'back', 'front']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# EV1 vs EV2 scatter (circle test)
ax = axes[0]
for vp in vp_order:
    m = valid_vps == vp
    ax.scatter(ev_lap1[m], ev_lap2[m], c=vp_colors_map[vp], s=3, alpha=0.3, label=vp)
ax.set_xlabel(f'Laplacian {EV_X_NAME}')
ax.set_ylabel(f'Laplacian {EV_Y_NAME}')
ax.set_title('Laplacian EVs: circle test')
ax.legend(markerscale=5, fontsize=8)
ax.set_aspect('equal')

# Theta distribution by viewpoint
ax = axes[1]
for vp in vp_order:
    m = valid_vps == vp
    ax.hist(theta_lap[m], bins=50, alpha=0.5, label=vp, color=vp_colors_map[vp], density=True)
ax.set_xlabel(f'atan2({EV_Y_NAME}, {EV_X_NAME})')
ax.set_ylabel('Density')
ax.set_title('Laplacian: angle distribution by VP')
ax.legend(fontsize=8)

# Boxplot
ax = axes[2]
bp_data_lap = [theta_lap[valid_vps == vp] for vp in unique_vps]
bp2 = ax.boxplot(bp_data_lap, labels=unique_vps, patch_artist=True)
for patch, vp in zip(bp2['boxes'], unique_vps):
    patch.set_facecolor(vp_colors_map[vp])
    patch.set_alpha(0.5)
ax.set_ylabel('atan2 angle')
ax.set_title('Laplacian: angle by viewpoint')

plt.suptitle('Angle Recovery from Graph Laplacian Eigenvectors', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# --- Quantitative AUC ---
from sklearn.metrics import roc_auc_score

N_sub = min(3000, len(valid_fvs))
rng_at = np.random.RandomState(42)
sub = rng_at.choice(len(valid_fvs), N_sub, replace=False)

th_sub = theta_lap[sub]
vp_sub = valid_vps[sub]
triu_at = np.triu_indices(N_sub, k=1)
same_vp_at = vp_sub[triu_at[0]] == vp_sub[triu_at[1]]
d_angle = np.abs(th_sub[triu_at[0]] - th_sub[triu_at[1]])
d_angle = np.minimum(d_angle, 2*np.pi - d_angle)
sim_angle = 1.0 - d_angle / np.pi
auc_lap = roc_auc_score(same_vp_at, sim_angle)
print(f'\nLaplacian atan2 viewpoint AUC = {auc_lap:.4f}')

In [ ]:
# ============================================================
# Multi-harmonic angle recovery from Laplacian eigenvectors
# ============================================================
# On a circle, Laplacian eigenfunctions come in pairs:
#   (cos kθ, sin kθ) for k = 1, 2, 3, ...
# Each pair gives an independent angle estimate.
# Combining multiple harmonics should give a cleaner estimate.
#
# Strategy: compute more Laplacian eigenvectors, identify
# pairs that trace circles, extract angle from each pair,
# combine via circular mean.

N_EIG = 20  # compute more eigenvectors

print(f'Computing {N_EIG} smallest Laplacian eigenvectors...')
eigenvalues_L, eigenvectors_L = smallest_eigenvectors(L_sym, n_eig=N_EIG)
print(f'  Eigenvalues: {eigenvalues_L}')

# --- Identify harmonic pairs ---
# On a circle, eigenvalues come in degenerate pairs (same value for cos kθ and sin kθ).
# Check which consecutive eigenvectors have similar eigenvalues.
print('\nEigenvalue gaps (looking for degenerate pairs):')
for i in range(1, N_EIG-1):
    gap_to_next = eigenvalues_L[i+1] - eigenvalues_L[i]
    gap_to_prev = eigenvalues_L[i] - eigenvalues_L[i-1]
    print(f'  EV {i:>2} (λ={eigenvalues_L[i]:.6f}): gap_prev={gap_to_prev:.6f}, gap_next={gap_to_next:.6f}')

# --- Visualize all consecutive pairs ---
n_pairs = min(8, (N_EIG - 1) // 2)
fig, axes = plt.subplots(2, n_pairs, figsize=(4*n_pairs, 8))

vp_colors_map = {'right': 'blue', 'left': 'red', 'front': 'green', 'back': 'orange'}
vp_order = ['right', 'left', 'back', 'front']

pair_angles = []  # store angles from each pair
pair_aucs = []    # store AUC from each pair

# Subsample for AUC computation
N_sub_mh = min(3000, len(valid_fvs))
rng_mh = np.random.RandomState(42)
sub_mh = rng_mh.choice(len(valid_fvs), N_sub_mh, replace=False)
vp_sub_mh = valid_vps[sub_mh]
triu_mh = np.triu_indices(N_sub_mh, k=1)
same_vp_mh = vp_sub_mh[triu_mh[0]] == vp_sub_mh[triu_mh[1]]

for pi in range(n_pairs):
    # Try consecutive pairs starting from EV1
    a_idx = 1 + 2 * pi
    b_idx = 2 + 2 * pi
    if b_idx >= N_EIG:
        break

    ev_a = eigenvectors_L[:, a_idx]
    ev_b = eigenvectors_L[:, b_idx]
    theta_pair = np.arctan2(ev_b, ev_a)
    pair_angles.append(theta_pair)

    # Scatter
    ax = axes[0, pi]
    for vp in vp_order:
        m = valid_vps == vp
        ax.scatter(ev_a[m], ev_b[m], c=vp_colors_map[vp], s=2, alpha=0.2)
    ax.set_xlabel(f'EV {a_idx}')
    ax.set_ylabel(f'EV {b_idx}')
    ax.set_title(f'λ={eigenvalues_L[a_idx]:.4f},{eigenvalues_L[b_idx]:.4f}')
    ax.set_aspect('equal')

    # Angle distribution
    ax = axes[1, pi]
    for vp in vp_order:
        m = valid_vps == vp
        ax.hist(theta_pair[m], bins=40, alpha=0.4, color=vp_colors_map[vp], density=True)
    ax.set_xlabel('atan2')
    ax.set_title(f'Pair ({a_idx},{b_idx})')

    # AUC
    th_sub = theta_pair[sub_mh]
    d_ang = np.abs(th_sub[triu_mh[0]] - th_sub[triu_mh[1]])
    d_ang = np.minimum(d_ang, 2*np.pi - d_ang)
    sim_ang = 1.0 - d_ang / np.pi
    auc_pair = roc_auc_score(same_vp_mh, sim_ang)
    pair_aucs.append(auc_pair)
    axes[0, pi].set_title(f'EV({a_idx},{b_idx}) AUC={auc_pair:.3f}\nλ={eigenvalues_L[a_idx]:.4f},{eigenvalues_L[b_idx]:.4f}')

plt.suptitle('Laplacian Eigenvector Pairs (potential harmonics)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('\nPer-pair AUC:')
for pi, auc in enumerate(pair_aucs):
    a_idx = 1 + 2 * pi
    b_idx = 2 + 2 * pi
    print(f'  Pair ({a_idx},{b_idx}): AUC={auc:.4f}')

# --- Combine harmonics via embedding distance ---
# Stack all eigenvector pairs into a multi-dimensional embedding.
# Distance in this space integrates information from all harmonics.
print('\n--- Combined multi-harmonic distance ---')

for n_use in [1, 2, 3, 4, 6, 8]:
    if 2 * n_use + 1 > N_EIG:
        break
    # Use eigenvectors 1..2*n_use (skip EV0 which is constant)
    emb = eigenvectors_L[sub_mh, 1:2*n_use+1]  # [N_sub, 2*n_use]
    # Euclidean distance in this space
    from scipy.spatial.distance import pdist, squareform
    d_emb = squareform(pdist(emb, 'euclidean'))
    sim_emb = -d_emb[triu_mh]  # negative distance as similarity
    auc_emb = roc_auc_score(same_vp_mh, sim_emb)
    print(f'  EVs 1..{2*n_use} ({n_use} pairs): AUC={auc_emb:.4f}')

# --- Also try cosine distance in eigenvector space ---
print('\n--- Cosine distance in eigenvector space ---')
for n_use in [1, 2, 3, 4, 6, 8]:
    if 2 * n_use + 1 > N_EIG:
        break
    emb = eigenvectors_L[sub_mh, 1:2*n_use+1]
    emb_n = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-10)
    sim_cos = emb_n @ emb_n.T
    auc_cos = roc_auc_score(same_vp_mh, sim_cos[triu_mh])
    print(f'  EVs 1..{2*n_use} ({n_use} pairs): AUC={auc_cos:.4f}')


In [ ]:
# ============================================================
# Viewpoint distance metric: 3D Laplacian embedding
# ============================================================
# Euclidean distance in EVs 1-3 (skipping constant EV0).
# EV4+ are nuisance-hijacked (lighting bottlenecks); see investigation cells.
# Full evaluation: classification, distributions, comparison.

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

BEST_N_EV = 3  # EVs 1..3 (EV4 is a lighting-session bottleneck)
emb_best = eigenvectors_L[:, 1:BEST_N_EV+1]  # [N, 4]

# --- 1. AUC comparison: Laplacian 3D vs original FV cosine ---
print('=== Viewpoint Similarity AUC ===')
N_sub_ev = min(3000, len(valid_fvs))
rng_ev = np.random.RandomState(42)
sub_ev = rng_ev.choice(len(valid_fvs), N_sub_ev, replace=False)
vp_sub_ev = valid_vps[sub_ev]
triu_ev = np.triu_indices(N_sub_ev, k=1)
same_vp_ev = vp_sub_ev[triu_ev[0]] == vp_sub_ev[triu_ev[1]]

# Laplacian 3D: negative euclidean distance as similarity
emb_sub = emb_best[sub_ev]
from scipy.spatial.distance import pdist, squareform
d_lap = squareform(pdist(emb_sub, 'euclidean'))
sim_lap = -d_lap[triu_ev]
auc_lap = roc_auc_score(same_vp_ev, sim_lap)

# Original FV cosine
fvs_sub_ev = valid_fvs[sub_ev]
sim_fv_ev = (fvs_sub_ev @ fvs_sub_ev.T)[triu_ev]
auc_fv_ev = roc_auc_score(same_vp_ev, sim_fv_ev)

# atan2 single angle (from selected EV pair)
th_single = np.arctan2(eigenvectors_L[:, EV_Y], eigenvectors_L[:, EV_X])
th_sub = th_single[sub_ev]
d_ang = np.abs(th_sub[triu_ev[0]] - th_sub[triu_ev[1]])
d_ang = np.minimum(d_ang, 2*np.pi - d_ang)
sim_ang = 1.0 - d_ang / np.pi
auc_ang = roc_auc_score(same_vp_ev, sim_ang)

print(f'  Laplacian 3D euclidean:  AUC = {auc_lap:.4f}')
print(f'  Laplacian atan2 angle:   AUC = {auc_ang:.4f}')
print(f'  Original FV cosine:      AUC = {auc_fv_ev:.4f}')

# --- 2. Distribution plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

MAX_PLOT = 200000
rng_p = np.random.RandomState(42)
idx_s = rng_p.choice(np.where(same_vp_ev)[0], min(MAX_PLOT, same_vp_ev.sum()), replace=False)
idx_d = rng_p.choice(np.where(~same_vp_ev)[0], min(MAX_PLOT, (~same_vp_ev).sum()), replace=False)

for ax, sims, title, auc_val in [
    (axes[0], sim_lap, 'Laplacian 3D (neg. euclidean)', auc_lap),
    (axes[1], sim_ang, 'Laplacian atan2 angle', auc_ang),
    (axes[2], sim_fv_ev, 'Original FV cosine', auc_fv_ev),
]:
    ax.hist(sims[idx_s], bins=50, alpha=0.5, density=True, label='Same VP', color='green')
    ax.hist(sims[idx_d], bins=50, alpha=0.5, density=True, label='Diff VP', color='red')
    ax.set_xlabel('Similarity')
    ax.set_ylabel('Density')
    ax.set_title(f'{title}\nAUC={auc_val:.4f}')
    ax.legend()

plt.suptitle('Viewpoint Similarity Distributions', fontsize=13)
plt.tight_layout()
plt.show()

# --- 3. kNN classification ---
print('\n=== kNN Classification: Laplacian 3D vs Original FV ===')
print(f'{"seeds":>6} {"lap3d":>8} {"fv_cos":>8} {"diff":>8}')

for n_seeds in [1, 2, 5, 10, 20, 50]:
    rng_kn = np.random.RandomState(RANDOM_SEED)
    si_kn = []
    for vp in unique_vps:
        vp_idx = np.where((valid_vps == vp) & is_pure_vp)[0]
        if len(vp_idx) == 0:
            vp_idx = np.where(valid_vps == vp)[0]
        ns = min(n_seeds, len(vp_idx))
        chosen = rng_kn.choice(vp_idx, size=ns, replace=False)
        si_kn.extend(chosen.tolist())

    train_kn = np.array(si_kn)
    test_kn = np.array([i for i in range(len(valid_fvs)) if i not in set(si_kn)])
    k_cls = min(5, len(train_kn))

    # Laplacian 3D
    clf_lap = KNeighborsClassifier(n_neighbors=k_cls, metric='euclidean')
    clf_lap.fit(emb_best[train_kn], valid_vps[train_kn])
    pred_lap = clf_lap.predict(emb_best[test_kn])
    acc_lap = (pred_lap == valid_vps[test_kn]).mean()

    # Original FV cosine
    clf_fv = KNeighborsClassifier(n_neighbors=k_cls, metric='cosine')
    clf_fv.fit(valid_fvs[train_kn], valid_vps[train_kn])
    pred_fv = clf_fv.predict(valid_fvs[test_kn])
    acc_fv = (pred_fv == valid_vps[test_kn]).mean()

    d = acc_lap - acc_fv
    print(f'{n_seeds:>6} {acc_lap:>8.4f} {acc_fv:>8.4f} {d:>+8.4f}')

# --- 4. Full classification report for best config ---
print('\n=== Classification Report: Laplacian 3D, seeds=10/vp ===')
rng_best = np.random.RandomState(RANDOM_SEED)
si_best = []
for vp in unique_vps:
    vp_idx = np.where((valid_vps == vp) & is_pure_vp)[0]
    if len(vp_idx) == 0:
        vp_idx = np.where(valid_vps == vp)[0]
    ns = min(10, len(vp_idx))
    chosen = rng_best.choice(vp_idx, size=ns, replace=False)
    si_best.extend(chosen.tolist())
train_best = np.array(si_best)
test_best = np.array([i for i in range(len(valid_fvs)) if i not in set(si_best)])

clf_report = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
clf_report.fit(emb_best[train_best], valid_vps[train_best])
pred_report = clf_report.predict(emb_best[test_best])
print(classification_report(valid_vps[test_best], pred_report, zero_division=0))

# --- Relaxed accuracy ---
# If raw VP is e.g. 'frontright' and prediction is 'front' or 'right', count as correct

raw_test = valid_raw_vps[test_best]
strict_correct = (pred_report == valid_vps[test_best])
relaxed_correct_arr = np.array([
    relaxed_correct(raw_test[i], pred_report[i])
    for i in range(len(pred_report))
])
strict_acc = strict_correct.mean()
relaxed_acc = relaxed_correct_arr.mean()
n_relaxed_extra = relaxed_correct_arr.sum() - strict_correct.sum()

print(f'\nStrict accuracy:  {strict_acc:.4f}')
print(f'Relaxed accuracy: {relaxed_acc:.4f} (+{n_relaxed_extra} from boundary VPs)')

# Per-VP relaxed breakdown
print(f'\nPer-viewpoint (strict -> relaxed):')
for vp in unique_vps:
    vp_m = valid_vps[test_best] == vp
    s = strict_correct[vp_m].sum()
    r = relaxed_correct_arr[vp_m].sum()
    t = vp_m.sum()
    print(f'  {vp:>10}: {s}/{t} ({s/t:.1%}) -> {r}/{t} ({r/t:.1%})')

# Confusion matrix
cm_lap = confusion_matrix(valid_vps[test_best], pred_report, labels=unique_vps)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm_lap, display_labels=unique_vps).plot(ax=ax, cmap='Blues', values_format='d')
acc_report = (pred_report == valid_vps[test_best]).mean()
ax.set_title(f'Laplacian 3D kNN (seeds=10/vp)\nAccuracy: {acc_report:.3f}')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Misclassification inspector (Laplacian 3D kNN)
# ============================================================
# Shows ONE random misclassified detection with detailed info:
# - Full image with GT bbox (yellow) and detection bbox (cyan)
# - Viewpoint scores, nearest neighbors info
# Re-run cell to see a different example.

from PIL import Image, ImageDraw

# Build GT annotation lookup (for GT bbox)
det_to_gt_annot = {m.detection_id: m.gt_annotation for m in matched}

# Pick a random misclassified detection
wrong_mask_lap = pred_report != valid_vps[test_best]
misclassed_idx_lap = np.where(wrong_mask_lap)[0]
misclassed_global_lap = test_best[misclassed_idx_lap]

print(f'Total misclassified: {len(misclassed_idx_lap)}/{len(test_best)}')
for vp in unique_vps:
    vp_m = valid_vps[test_best] == vp
    vp_w = wrong_mask_lap & vp_m
    print(f'  {vp:>10}: {vp_w.sum()}/{vp_m.sum()} wrong ({vp_w.sum()/max(vp_m.sum(),1):.1%})')

if len(misclassed_idx_lap) > 0:
    # Random selection (changes each run)
    pick = np.random.randint(len(misclassed_idx_lap))
    ti = misclassed_idx_lap[pick]   # index into test arrays
    gi = misclassed_global_lap[pick]  # index into valid_*
    det_id = valid_dets[gi]
    true_vp_coarse = valid_vps[gi]
    true_vp_raw = valid_raw_vps[gi]
    pred_vp = pred_report[ti]

    # Laplacian embedding of this point
    emb_point = emb_best[gi]

    # Distance to all training points, grouped by VP
    train_embs = emb_best[train_best]
    train_vps_arr = valid_vps[train_best]
    dists_to_train = np.linalg.norm(train_embs - emb_point[None, :], axis=1)

    # Per-VP: nearest training point distance
    print(f'\n--- Detection {det_id} ---')
    print(f'True VP: {true_vp_raw} (coarse: {true_vp_coarse}), Predicted VP: {pred_vp}')
    is_relaxed_correct = relaxed_correct(true_vp_raw, pred_vp)
    print(f'Relaxed correct: {is_relaxed_correct}')
    print(f'Laplacian embedding: {emb_point}')
    print(f'\nDistance to nearest seed of each VP:')
    for vp in unique_vps:
        vp_mask_train = train_vps_arr == vp
        if vp_mask_train.any():
            min_dist = dists_to_train[vp_mask_train].min()
            mean_dist = dists_to_train[vp_mask_train].mean()
            marker = ' <-- TRUE' if vp == true_vp_coarse else (' <-- PRED' if vp == pred_vp else '')
            print(f'  {vp:>10}: nearest={min_dist:.6f}, mean={mean_dist:.6f}{marker}')

    # 5 nearest training neighbors
    nn_order = np.argsort(dists_to_train)[:5]
    print(f'\n5 nearest training neighbors:')
    for rank, ni in enumerate(nn_order):
        print(f'  #{rank+1}: VP={train_vps_arr[ni]}, dist={dists_to_train[ni]:.6f}')

    # atan2 angle
    angle = np.degrees(np.arctan2(eigenvectors_L[gi, 2], eigenvectors_L[gi, 1]))
    print(f'\natan2 angle: {angle:.1f}°')

    # Load image and draw bboxes
    det_obj = dataset.get_detection(det_id)
    img = Image.open(det_obj.image_path).convert('RGB')
    img_w, img_h = img.size

    # Detection bbox (cyan)
    sx1, sy1, sx2, sy2 = det_obj.bbox.int().tolist()
    sx1, sy1 = max(0, sx1), max(0, sy1)
    sx2, sy2 = min(img_w, sx2), min(img_h, sy2)

    # GT bbox (yellow)
    gt_ann = det_to_gt_annot.get(det_id)
    if gt_ann is not None:
        gx1, gy1, gx2, gy2 = int(gt_ann.bbox.x1), int(gt_ann.bbox.y1), \
                              int(gt_ann.bbox.x2), int(gt_ann.bbox.y2)
    else:
        gx1, gy1, gx2, gy2 = sx1, sy1, sx2, sy2

    # Crop to union of both bboxes with padding
    ux1 = min(sx1, gx1)
    uy1 = min(sy1, gy1)
    ux2 = max(sx2, gx2)
    uy2 = max(sy2, gy2)
    pad = int(max(ux2 - ux1, uy2 - uy1) * 0.15)
    cx1 = max(0, ux1 - pad)
    cy1 = max(0, uy1 - pad)
    cx2 = min(img_w, ux2 + pad)
    cy2 = min(img_h, uy2 + pad)

    crop = img.crop((cx1, cy1, cx2, cy2)).copy()
    draw = ImageDraw.Draw(crop)

    # Draw detection bbox (cyan)
    draw.rectangle([sx1-cx1, sy1-cy1, sx2-cx1, sy2-cy1], outline='cyan', width=3)
    # Draw GT bbox (yellow)
    draw.rectangle([gx1-cx1, gy1-cy1, gx2-cx1, gy2-cy1], outline='yellow', width=3)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(crop)
    ax.set_title(
        f'TRUE: {true_vp_raw} (coarse: {true_vp_coarse})  |  PRED: {pred_vp}  |  angle: {angle:.1f}°\n'
        f'cyan=detection bbox, yellow=GT bbox'
        + ('  [relaxed OK]' if is_relaxed_correct else ''),
        fontsize=12, color='orange' if is_relaxed_correct else 'red'
    )
    ax.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# Visibility score vs hard misclassification
# ============================================================
# Visibility = fraction of active components (weight > 0)
# Hypothesis: hard errors concentrate on low-visibility detections.

# Compute active components from raw FVs (weight dim is first of each block)
raw_fvs_valid = all_fvs_raw[valid_fv_idx]  # [N, K*block]
w_idx_vis = np.arange(K) * block
active_components = raw_fvs_valid[:, w_idx_vis] > 0  # [N, K] bool
visibility = active_components.sum(axis=1) / K  # fraction of K components active

print(f'Visibility score: min={visibility.min():.3f}, max={visibility.max():.3f}, '
      f'mean={visibility.mean():.3f}, median={np.median(visibility):.3f}')

# Hard errors from Laplacian 3D kNN (computed in the distance metric cell)
raw_test_vis = valid_raw_vps[test_best]
strict_wrong_vis = pred_report != valid_vps[test_best]
relaxed_vis = np.array([relaxed_correct(raw_test_vis[i], pred_report[i])
                         for i in range(len(pred_report))])
hard_wrong_vis = strict_wrong_vis & ~relaxed_vis

vis_all = visibility[test_best]
vis_hard = visibility[test_best[hard_wrong_vis]]
vis_correct = visibility[test_best[~strict_wrong_vis]]

print(f'\nAll test: n={len(vis_all)}, mean vis={vis_all.mean():.3f}')
print(f'Correct:  n={len(vis_correct)}, mean vis={vis_correct.mean():.3f}')
print(f'Hard err: n={len(vis_hard)}, mean vis={vis_hard.mean():.3f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram: all vs hard errors
ax = axes[0]
ax.hist(vis_all, bins=40, alpha=0.4, density=True, color='blue', label=f'All test (n={len(vis_all)})')
ax.hist(vis_hard, bins=40, alpha=0.7, density=True, color='red', label=f'Hard errors (n={len(vis_hard)})')
ax.set_xlabel('Visibility (fraction of active components)')
ax.set_ylabel('Density')
ax.set_title('Visibility distribution: all vs hard errors')
ax.legend()

# Error rate by visibility bin
ax = axes[1]
bins = np.linspace(visibility.min(), visibility.max(), 20)
bin_idx = np.digitize(vis_all, bins) - 1
bin_centers = []
error_rates = []
bin_counts = []
for b in range(len(bins) - 1):
    mask_b = bin_idx == b
    if mask_b.sum() < 5:
        continue
    err_rate = hard_wrong_vis[mask_b].mean() if mask_b.sum() > 0 else 0
    bin_centers.append((bins[b] + bins[b+1]) / 2)
    error_rates.append(err_rate)
    bin_counts.append(mask_b.sum())

ax.bar(bin_centers, error_rates, width=(bins[1]-bins[0])*0.8, color='red', alpha=0.6)
ax.set_xlabel('Visibility (fraction of active components)')
ax.set_ylabel('Hard error rate')
ax.set_title('Error rate by visibility')

for x, y, n in zip(bin_centers, error_rates, bin_counts):
    if y > 0.01:
        ax.text(x, y + 0.01, f'n={n}', ha='center', fontsize=7)

plt.suptitle('Does partial visibility cause misclassification?', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Is visibility an independent axis in the Laplacian embedding?
# ============================================================
# Check correlation of each Laplacian eigenvector with the
# visibility score. If some eigenvectors correlate with visibility,
# it's a separate axis from viewpoint.

from scipy.stats import pearsonr, spearmanr

# Use the 20 eigenvectors from cell 7 (multi_harmonic_angle)
# eigenvectors_L and eigenvalues_L have 20 components

print('Correlation of Laplacian eigenvectors with visibility:')
print(f'{"EV":>4} {"eigenval":>10} {"pearson_r":>10} {"p_val":>10} {"spearman":>10}')

ev_vis_corrs = []
for i in range(min(eigenvectors_L.shape[1], 20)):
    ev = eigenvectors_L[:, i]
    r_p, p_p = pearsonr(ev, visibility)
    r_s, p_s = spearmanr(ev, visibility)
    ev_vis_corrs.append((i, eigenvalues_L[i] if i < len(eigenvalues_L) else 0, r_p, p_p, r_s))
    sig = '***' if p_p < 0.001 else '**' if p_p < 0.01 else '*' if p_p < 0.05 else ''
    print(f'{i:>4} {eigenvalues_L[i] if i < len(eigenvalues_L) else 0:>10.6f} {r_p:>+10.4f} {p_p:>10.2e} {r_s:>+10.4f} {sig}')

# Also check: viewpoint correlation for comparison
# Encode viewpoint as numeric (right=0, backright=45, front=90, etc.)
# Simple: just use the atan2 angle as viewpoint proxy
theta_all = np.arctan2(eigenvectors_L[:, EV_Y], eigenvectors_L[:, EV_X])

print(f'\nCorrelation of visibility with atan2 viewpoint angle:')
r_vis_angle, p_vis_angle = pearsonr(visibility, theta_all)
print(f'  Pearson: {r_vis_angle:+.4f} (p={p_vis_angle:.2e})')
r_vis_angle_s, _ = spearmanr(visibility, theta_all)
print(f'  Spearman: {r_vis_angle_s:+.4f}')

# Scatter plots: top correlated EVs vs visibility
# Sort by absolute pearson correlation
sorted_corrs = sorted(ev_vis_corrs[1:], key=lambda x: abs(x[2]), reverse=True)  # skip EV0
top_vis_evs = sorted_corrs[:4]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
vp_colors_map = {'right': 'blue', 'left': 'red', 'front': 'green', 'back': 'orange'}

for ax, (ev_idx, ev_val, r_p, p_p, r_s) in zip(axes, top_vis_evs):
    for vp in ['right', 'left', 'back', 'front']:
        m = valid_vps == vp
        ax.scatter(eigenvectors_L[m, ev_idx], visibility[m],
                   c=vp_colors_map[vp], s=3, alpha=0.2, label=vp)
    ax.set_xlabel(f'EV {ev_idx}')
    ax.set_ylabel('Visibility')
    ax.set_title(f'EV {ev_idx} (r={r_p:+.3f}, p={p_p:.1e})')
    ax.legend(markerscale=5, fontsize=8)

plt.suptitle('Laplacian eigenvectors most correlated with visibility', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Also: 2D scatter of viewpoint angle vs visibility, colored by VP
fig, ax = plt.subplots(figsize=(10, 6))
for vp in ['right', 'left', 'back', 'front']:
    m = valid_vps == vp
    ax.scatter(theta_all[m], visibility[m],
               c=vp_colors_map[vp], s=5, alpha=0.2, label=f'{vp} ({m.sum()})')
ax.set_xlabel('atan2 viewpoint angle')
ax.set_ylabel('Visibility score')
ax.set_title('Viewpoint angle vs Visibility')
ax.legend(markerscale=5)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Regress out visibility from eigenvectors
# ============================================================
# EV1 is 84% visibility. Regress out the linear visibility
# component from each eigenvector, keep residuals.
# The residuals should contain pure viewpoint info.

from scipy.spatial.distance import pdist, squareform

# Regress visibility out of each eigenvector
# For each EV_i: EV_i_clean = EV_i - (EV_i . vis / vis . vis) * vis
vis_centered = visibility - visibility.mean()
vis_norm_sq = np.dot(vis_centered, vis_centered)

evecs_clean = np.zeros_like(eigenvectors_L)
print('Regressing out visibility from each eigenvector:')
for i in range(eigenvectors_L.shape[1]):
    ev = eigenvectors_L[:, i]
    # Project out visibility component
    coeff = np.dot(ev, vis_centered) / vis_norm_sq
    evecs_clean[:, i] = ev - coeff * vis_centered
    
    # Check residual correlation
    r_before = np.corrcoef(ev, visibility)[0, 1]
    r_after = np.corrcoef(evecs_clean[:, i], visibility)[0, 1]
    print(f'  EV {i:>2}: r_vis {r_before:+.4f} -> {r_after:+.4f}')

# --- Evaluate different EV selections with cleaned eigenvectors ---
N_sub_cl = min(3000, len(valid_fvs))
rng_cl = np.random.RandomState(42)
sub_cl = rng_cl.choice(len(valid_fvs), N_sub_cl, replace=False)
vp_sub_cl = valid_vps[sub_cl]
triu_cl = np.triu_indices(N_sub_cl, k=1)
same_vp_cl = vp_sub_cl[triu_cl[0]] == vp_sub_cl[triu_cl[1]]

print(f'\n{"selection":>25} {"AUC_orig":>9} {"AUC_clean":>10} {"diff":>7}')

for label, ev_indices in [
    ('EVs 1-2', [1, 2]),
    ('EVs 1-3', [1, 2, 3]),
    ('EVs 1-6', [1, 2, 3, 4, 5, 6]),
    ('EVs 1-8', [1, 2, 3, 4, 5, 6, 7, 8]),
    ('EVs 1-10', list(range(1, 11))),
]:
    emb_orig = eigenvectors_L[sub_cl][:, ev_indices]
    d_orig = squareform(pdist(emb_orig, 'euclidean'))
    auc_orig = roc_auc_score(same_vp_cl, -d_orig[triu_cl])

    emb_clean = evecs_clean[sub_cl][:, ev_indices]
    d_clean = squareform(pdist(emb_clean, 'euclidean'))
    auc_clean = roc_auc_score(same_vp_cl, -d_clean[triu_cl])

    print(f'{label:>25} {auc_orig:>9.4f} {auc_clean:>10.4f} {auc_clean-auc_orig:>+7.4f}')

# --- kNN classification with cleaned EVs 1-3 ---
print(f'\nkNN classification (EVs 1-3):')
print(f'{"seeds":>6} {"orig":>8} {"clean":>8} {"diff":>8}')

for n_seeds in [5, 10, 20, 50]:
    rng_kn = np.random.RandomState(RANDOM_SEED)
    si_kn = []
    for vp in unique_vps:
        vp_idx = np.where((valid_vps == vp) & is_pure_vp)[0]
        if len(vp_idx) == 0: vp_idx = np.where(valid_vps == vp)[0]
        si_kn.extend(rng_kn.choice(vp_idx, size=min(n_seeds, len(vp_idx)), replace=False).tolist())
    tr, te = np.array(si_kn), np.array([i for i in range(len(valid_fvs)) if i not in set(si_kn)])

    for emb, label in [(eigenvectors_L[:, 1:4], 'orig'), (evecs_clean[:, 1:4], 'clean')]:
        clf = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
        clf.fit(emb[tr], valid_vps[tr])
        pred = clf.predict(emb[te])
        if label == 'orig':
            acc_orig = (pred == valid_vps[te]).mean()
        else:
            acc_clean = (pred == valid_vps[te]).mean()

    print(f'{n_seeds:>6} {acc_orig:>8.4f} {acc_clean:>8.4f} {acc_clean-acc_orig:>+8.4f}')

# Full report for cleaned
rng_r = np.random.RandomState(RANDOM_SEED)
si_r = []
for vp in unique_vps:
    vp_idx = np.where((valid_vps == vp) & is_pure_vp)[0]
    if len(vp_idx) == 0: vp_idx = np.where(valid_vps == vp)[0]
    si_r.extend(rng_r.choice(vp_idx, size=min(10, len(vp_idx)), replace=False).tolist())
tr_r, te_r = np.array(si_r), np.array([i for i in range(len(valid_fvs)) if i not in set(si_r)])
clf_r = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
clf_r.fit(evecs_clean[tr_r, 1:4], valid_vps[tr_r])
pred_cl = clf_r.predict(evecs_clean[te_r, 1:4])

print(f'\n=== Cleaned EVs 1-3, seeds=10/vp ===')
print(classification_report(valid_vps[te_r], pred_cl, zero_division=0))
raw_te_cl = valid_raw_vps[te_r]
strict_cl = (pred_cl == valid_vps[te_r]).mean()
relaxed_cl = np.array([relaxed_correct(raw_te_cl[i], pred_cl[i])
                        for i in range(len(pred_cl))]).mean()
print(f'Strict: {strict_cl:.4f}, Relaxed: {relaxed_cl:.4f}')

# --- Hard errors comparison ---
strict_wrong_cl = pred_cl != valid_vps[te_r]
relaxed_cl_arr = np.array([relaxed_correct(raw_te_cl[i], pred_cl[i])
                            for i in range(len(pred_cl))])
hard_wrong_cl = strict_wrong_cl & ~relaxed_cl_arr
print(f'Hard errors: {hard_wrong_cl.sum()} (original: 1485)')

# Error rate by visibility for cleaned vs original
vis_te = visibility[te_r]
bins = [0, 0.3, 0.5, 0.7, 1.01]
print(f'\nError rate by visibility band:')
print(f'{"vis_band":>12} {"orig_err":>10} {"clean_err":>10}')
for i in range(len(bins)-1):
    band = (vis_te >= bins[i]) & (vis_te < bins[i+1])
    if band.sum() == 0:
        continue
    # Original
    clf_o = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
    clf_o.fit(eigenvectors_L[tr_r, 1:4], valid_vps[tr_r])
    pred_o = clf_o.predict(eigenvectors_L[te_r, 1:4])
    hw_o = ((pred_o != valid_vps[te_r]) & ~np.array([relaxed_correct(raw_te_cl[j], pred_o[j]) for j in range(len(pred_o))]))[band]
    # Cleaned
    hw_c = hard_wrong_cl[band]
    print(f'  [{bins[i]:.1f}-{bins[i+1]:.1f}): n={band.sum():>5}, orig={hw_o.mean():.3f}, clean={hw_c.mean():.3f}, diff={hw_c.mean()-hw_o.mean():+.3f}')


In [ ]:
# ============================================================
# Angle distribution: original vs cleaned eigenvectors
# ============================================================

theta_orig = np.arctan2(eigenvectors_L[:, EV_Y], eigenvectors_L[:, EV_X])
theta_clean = np.arctan2(evecs_clean[:, EV_Y], evecs_clean[:, EV_X])

vp_colors_map = {'right': 'blue', 'left': 'red', 'front': 'green', 'back': 'orange'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top row: original
ax = axes[0, 0]
for vp in ['right', 'left', 'back', 'front']:
    m = valid_vps == vp
    ax.scatter(eigenvectors_L[m, EV_X], eigenvectors_L[m, EV_Y],
               c=vp_colors_map[vp], s=3, alpha=0.3, label=vp)
ax.set_xlabel(f'{EV_X_NAME} (original)')
ax.set_ylabel(f'{EV_Y_NAME} (original)')
ax.set_title(f'Original: {EV_X_NAME} vs {EV_Y_NAME}')
ax.legend(markerscale=5, fontsize=8)
ax.set_aspect('equal')

ax = axes[0, 1]
for vp in ['right', 'left', 'back', 'front']:
    m = valid_vps == vp
    ax.hist(theta_orig[m], bins=50, alpha=0.4, density=True, color=vp_colors_map[vp], label=vp)
ax.set_xlabel('atan2 angle (original)')
ax.set_ylabel('Density')
ax.set_title('Original angle distribution')
ax.legend(fontsize=8)

# Bottom row: cleaned
ax = axes[1, 0]
for vp in ['right', 'left', 'back', 'front']:
    m = valid_vps == vp
    ax.scatter(evecs_clean[m, EV_X], evecs_clean[m, EV_Y],
               c=vp_colors_map[vp], s=3, alpha=0.3, label=vp)
ax.set_xlabel(f'{EV_X_NAME} cleaned')
ax.set_ylabel(f'{EV_Y_NAME} cleaned')
ax.set_title(f'Cleaned: {EV_X_NAME} vs {EV_Y_NAME}')
ax.legend(markerscale=5, fontsize=8)
ax.set_aspect('equal')

ax = axes[1, 1]
for vp in ['right', 'left', 'back', 'front']:
    m = valid_vps == vp
    ax.hist(theta_clean[m], bins=50, alpha=0.4, density=True, color=vp_colors_map[vp], label=vp)
ax.set_xlabel('atan2 angle (cleaned)')
ax.set_ylabel('Density')
ax.set_title('Cleaned angle distribution')
ax.legend(fontsize=8)

# Hard errors overlay
hard_wrong_orig_ang = ((pred_report != valid_vps[test_best]) &
                       ~np.array([relaxed_correct(valid_raw_vps[test_best][i], pred_report[i])
                                  for i in range(len(pred_report))]))
hard_wrong_cl_ang = ((pred_cl != valid_vps[te_r]) & 
                     ~np.array([relaxed_correct(raw_te_cl[i], pred_cl[i])
                                for i in range(len(pred_cl))]))

axes[0, 1].hist(theta_orig[test_best[hard_wrong_orig_ang]], bins=50, alpha=0.8, density=True,
                color='black', histtype='step', linewidth=2.5, label=f'hard errors ({hard_wrong_orig_ang.sum()})')
axes[0, 1].legend(fontsize=7)

axes[1, 1].hist(theta_clean[te_r[hard_wrong_cl_ang]], bins=50, alpha=0.8, density=True,
                color='black', histtype='step', linewidth=2.5, label=f'hard errors ({hard_wrong_cl_ang.sum()})')
axes[1, 1].legend(fontsize=7)

plt.suptitle('Original vs Visibility-Cleaned Eigenvectors', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# AUC from atan2 angle: original vs cleaned
N_sub_ang = min(3000, len(valid_fvs))
rng_ang = np.random.RandomState(42)
sub_ang = rng_ang.choice(len(valid_fvs), N_sub_ang, replace=False)
vp_sub_ang = valid_vps[sub_ang]
triu_ang = np.triu_indices(N_sub_ang, k=1)
same_vp_ang = vp_sub_ang[triu_ang[0]] == vp_sub_ang[triu_ang[1]]

for name, theta in [('Original', theta_orig), ('Cleaned', theta_clean)]:
    th_sub = theta[sub_ang]
    d_ang = np.abs(th_sub[triu_ang[0]] - th_sub[triu_ang[1]])
    d_ang = np.minimum(d_ang, 2*np.pi - d_ang)
    sim_ang = 1.0 - d_ang / np.pi
    auc_ang = roc_auc_score(same_vp_ang, sim_ang)
    print(f'{name} atan2 AUC: {auc_ang:.4f}')

# Correlation of cleaned angle with visibility
r_orig, _ = pearsonr(theta_orig, visibility)
r_clean, _ = pearsonr(theta_clean, visibility)
print(f'\nAngle-visibility correlation: original={r_orig:.4f}, cleaned={r_clean:.4f}')


In [ ]:
# ============================================================
# Angular analysis: do mixed viewpoints interpolate?
# ============================================================
# Use circular statistics (not linear boxplots).

# Angle from selected EV pair
theta_12 = np.arctan2(eigenvectors_L[:, EV_Y], eigenvectors_L[:, EV_X])
# Possible second angle from EVs 2,3
theta_23 = np.arctan2(eigenvectors_L[:, 3], eigenvectors_L[:, 2])

fg_counts = Counter(valid_raw_vps)
fg_sorted = sorted(fg_counts.keys(), key=lambda v: -fg_counts[v])
fg_sorted = [v for v in fg_sorted if fg_counts[v] >= 5]

# circular_mean imported from src.utils.circular_stats

# circular_std imported from src.utils.circular_stats

# circular_midpoint imported from src.utils.circular_stats

# Per-viewpoint circular stats
print(f'{"viewpoint":>15} {"N":>5} {"angle1":>10} {"std1":>7} {"angle2":>10} {"std2":>7} {"vis":>6}')
vp_angles = {}
for vp in fg_sorted:
    m = valid_raw_vps == vp
    m12 = circular_mean(theta_12[m])
    s12 = circular_std(theta_12[m])
    m23 = circular_mean(theta_23[m])
    s23 = circular_std(theta_23[m])
    vis_m = visibility[m].mean()
    vp_angles[vp] = (m12, s12, m23, s23, vis_m)
    print(f'{vp:>15} {m.sum():>5} {np.degrees(m12):>+8.1f}° {np.degrees(s12):>5.1f}° {np.degrees(m23):>+8.1f}° {np.degrees(s23):>5.1f}° {vis_m:>6.2f}')

# --- Polar plot of angle 1 ---
fg_colors_plot = {
    'right': 'blue', 'backright': 'cornflowerblue', 'frontright': 'deepskyblue',
    'left': 'red', 'backleft': 'salmon', 'frontleft': 'lightcoral',
    'back': 'orange', 'front': 'green',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': 'polar'})

for ax, theta, title, mean_idx in [
    (axes[0], theta_12, f'Angle 1: atan2({EV_Y_NAME},{EV_X_NAME})', 0),
    (axes[1], theta_23, 'Angle 2: atan2(EV4,EV3)', 2),
]:
    for vp in fg_sorted:
        m = valid_raw_vps == vp
        c = fg_colors_plot.get(vp, 'gray')
        # Plot individual points at radius = 1 + some jitter for density
        r = np.ones(m.sum()) + np.random.RandomState(42).uniform(-0.3, 0.3, m.sum())
        ax.scatter(theta[m], r, c=c, s=3, alpha=0.3)
    
    # Plot mean directions as arrows
    for vp in fg_sorted:
        mean_angle = vp_angles[vp][mean_idx]
        c = fg_colors_plot.get(vp, 'gray')
        ax.annotate('', xy=(mean_angle, 1.5), xytext=(mean_angle, 0),
                    arrowprops=dict(arrowstyle='->', color=c, lw=2))
        ax.text(mean_angle, 1.7, vp[:6], ha='center', va='center', fontsize=7, color=c)
    
    ax.set_title(title, fontsize=11, pad=20)
    ax.set_ylim(0, 2)
    ax.set_rticks([])

plt.suptitle('Polar angle distributions by viewpoint', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

# --- 2D angle space (angle1 vs angle2) ---
fig, ax = plt.subplots(figsize=(10, 8))
for vp in fg_sorted:
    m12, s12, m34, s34, _ = vp_angles[vp]
    c = fg_colors_plot.get(vp, 'gray')
    ax.errorbar(np.degrees(m12), np.degrees(m34),
                xerr=min(np.degrees(s12), 90), yerr=min(np.degrees(s34), 90),
                fmt='o', color=c, markersize=10, capsize=4, label=f'{vp} ({fg_counts[vp]})',
                linewidth=2)
ax.set_xlabel(f'Angle 1: atan2({EV_Y_NAME}, {EV_X_NAME}) [degrees]')
ax.set_ylabel('Angle 2: atan2(EV3, EV2) [degrees]')
ax.set_title('2D angular viewpoint space')
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Interpolation check ---
print('\n=== Interpolation check (circular midpoint) ===')
for compound, pure1, pure2 in [
    ('backright', 'back', 'right'),
    ('frontright', 'front', 'right'),
    ('backleft', 'back', 'left'),
    ('frontleft', 'front', 'left'),
]:
    if compound in vp_angles and pure1 in vp_angles and pure2 in vp_angles:
        a1_comp = vp_angles[compound][0]
        a1_p1 = vp_angles[pure1][0]
        a1_p2 = vp_angles[pure2][0]
        mid1 = circular_midpoint(a1_p1, a1_p2)
        
        a2_comp = vp_angles[compound][2]
        a2_p1 = vp_angles[pure1][2]
        a2_p2 = vp_angles[pure2][2]
        mid2 = circular_midpoint(a2_p1, a2_p2)
        
        print(f'  {compound:>12}:')
        print(f'    angle1: actual={np.degrees(a1_comp):+.1f}°, '
              f'{pure1}={np.degrees(a1_p1):+.1f}°, {pure2}={np.degrees(a1_p2):+.1f}°, '
              f'midpoint={np.degrees(mid1):+.1f}°')
        print(f'    angle2: actual={np.degrees(a2_comp):+.1f}°, '
              f'{pure1}={np.degrees(a2_p1):+.1f}°, {pure2}={np.degrees(a2_p2):+.1f}°, '
              f'midpoint={np.degrees(mid2):+.1f}°')


In [ ]:
# ============================================================
# 2D angular space — full scatter + per-viewpoint density
# ============================================================
# theta1 = atan2(EV2, EV1), theta2 = atan2(EV3, EV2)
# Shows the full ~7k detection distribution (not just mean+std).

from scipy.stats import gaussian_kde

theta1_full = np.degrees(np.arctan2(eigenvectors_L[:, EV_Y], eigenvectors_L[:, EV_X]))
theta2_full = np.degrees(np.arctan2(eigenvectors_L[:, 3], eigenvectors_L[:, 2]))

fg_colors_sc = {
    'right':      'royalblue',
    'backright':  'navy',
    'frontright': 'deepskyblue',
    'left':       'red',
    'backleft':   'darkred',
    'frontleft':  'hotpink',
    'back':       'orange',
    'front':      'limegreen',
}
fg_counts_sc = Counter(valid_raw_vps)
fg_sorted_sc = [v for v, c in fg_counts_sc.most_common() if c >= 5]

# --- Main panel: all VPs overlaid ---
fig, ax = plt.subplots(figsize=(13, 11))
for vp in fg_sorted_sc:
    m = valid_raw_vps == vp
    ax.scatter(theta1_full[m], theta2_full[m],
               c=fg_colors_sc.get(vp, 'gray'), s=25, alpha=0.55,
               label=f'{vp} ({m.sum()})', edgecolors='none')

ax.set_xlabel(f'Angle 1: atan2({EV_Y_NAME}, {EV_X_NAME}) [degrees]')
ax.set_ylabel('Angle 2: atan2(EV3, EV2) [degrees]')
ax.set_title('2D angular viewpoint space — all detections')
ax.legend(fontsize=10, ncol=2, loc='lower right', markerscale=2)
ax.grid(True, alpha=0.3)
ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
plt.tight_layout()
plt.show()

# --- Small multiples: one subplot per VP, highlighted against gray background ---
KDE_MIN_POINTS = 30
ncols = 4
nrows = int(np.ceil(len(fg_sorted_sc) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 4.5*nrows), sharex=True, sharey=True)
axes = np.atleast_2d(axes).ravel()

xgrid = np.linspace(-180, 180, 80)
ygrid = np.linspace(-180, 180, 80)
Xg, Yg = np.meshgrid(xgrid, ygrid)
grid_pts = np.vstack([Xg.ravel(), Yg.ravel()])

for ax_i, vp in enumerate(fg_sorted_sc):
    ax = axes[ax_i]
    ax.scatter(theta1_full, theta2_full, c='lightgray', s=8, alpha=0.4, edgecolors='none')
    m = valid_raw_vps == vp
    ax.scatter(theta1_full[m], theta2_full[m],
               c=fg_colors_sc.get(vp, 'gray'), s=25, alpha=0.75, edgecolors='none')
    if m.sum() >= KDE_MIN_POINTS:
        xy = np.vstack([theta1_full[m], theta2_full[m]])
        kde = gaussian_kde(xy, bw_method=0.25)
        Z = kde(grid_pts).reshape(Xg.shape)
        ax.contour(Xg, Yg, Z, levels=5, colors='black', linewidths=1.0, alpha=0.7)
    ax.set_title(f'{vp} (n={m.sum()})', fontsize=11)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-180, 180)
    ax.grid(True, alpha=0.3)

for k in range(len(fg_sorted_sc), len(axes)):
    axes[k].axis('off')

for ax in axes[-ncols:]:
    ax.set_xlabel('Angle 1 [°]')
for ax in axes[::ncols]:
    ax.set_ylabel('Angle 2 [°]')

plt.suptitle('Per-viewpoint distribution with KDE contours', fontsize=14, y=1.005)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Torus view: (angle1, angle2) tiled to reveal periodicity
# ============================================================
# Both angles are circular. Tile the [-180, 180]^2 fundamental
# domain 3x3 to visualize wrap-around and see if clusters form
# a connected loop across the boundary.

import matplotlib.patches as mpatches

fg_colors_t = fg_colors_sc  # defined in the scatter-detail cell

fig, ax = plt.subplots(figsize=(13, 13))

for dx in [-360, 0, 360]:
    for dy in [-360, 0, 360]:
        is_center = (dx == 0 and dy == 0)
        alpha = 0.65 if is_center else 0.15
        size  = 22 if is_center else 10
        for vp in fg_sorted_sc:
            m = valid_raw_vps == vp
            ax.scatter(theta1_full[m] + dx, theta2_full[m] + dy,
                       c=fg_colors_t.get(vp, 'gray'), s=size, alpha=alpha,
                       edgecolors='none')

# Dashed grid at ±180 boundaries
for k in range(4):
    x0 = -540 + k * 360
    ax.axvline(x0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axhline(x0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)

# Highlight fundamental domain
rect = mpatches.Rectangle((-180, -180), 360, 360, linewidth=2.5,
                          edgecolor='black', facecolor='none')
ax.add_patch(rect)

handles = [plt.Line2D([], [], marker='o', color=fg_colors_t[v], linestyle='',
                      markersize=10, label=v) for v in fg_sorted_sc]
ax.legend(handles=handles, fontsize=10, ncol=2, loc='upper left',
          bbox_to_anchor=(1.02, 1))

ax.set_xlabel(f'Angle 1: atan2({EV_Y_NAME}, {EV_X_NAME}) [degrees]')
ax.set_ylabel('Angle 2: atan2(EV3, EV2) [degrees]')
ax.set_title('Toroidal view — fundamental domain tiled 3x3\n(dashed lines = ±180° wrap)')
ax.set_xlim(-540, 540)
ax.set_ylim(-540, 540)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# --- Side-by-side polar: each angle on its own unit circle ---
fig, axes = plt.subplots(1, 2, figsize=(15, 7.5), subplot_kw={'projection': 'polar'})

rng_polar = np.random.RandomState(42)
for ax, theta_rad, title in [
    (axes[0], np.arctan2(eigenvectors_L[:, EV_Y], eigenvectors_L[:, EV_X]), f'Angle 1: atan2({EV_Y_NAME}, {EV_X_NAME})'),
    (axes[1], np.arctan2(eigenvectors_L[:, 3], eigenvectors_L[:, 2]), 'Angle 2: atan2(EV3, EV2)'),
]:
    for vp in fg_sorted_sc:
        m = valid_raw_vps == vp
        r = np.ones(m.sum()) + rng_polar.uniform(-0.25, 0.25, m.sum())
        ax.scatter(theta_rad[m], r, c=fg_colors_t.get(vp, 'gray'),
                   s=20, alpha=0.6, edgecolors='none')
    ax.set_ylim(0, 2)
    ax.set_rticks([])
    ax.set_title(title, pad=15, fontsize=12)

handles2 = [plt.Line2D([], [], marker='o', color=fg_colors_t[v], linestyle='',
                       markersize=10, label=v) for v in fg_sorted_sc]
fig.legend(handles=handles2, fontsize=10, ncol=4, loc='lower center',
           bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Per-angle polar distributions', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Interactive 3D scatter in (EV1, EV2, EV3) eigenvector space
# ============================================================
# Drag to rotate, scroll to zoom. Toggle classes via legend.

import plotly.graph_objects as go

# Optional subsample for snappier interaction
MAX_POINTS_3D = 6000
if len(valid_raw_vps) > MAX_POINTS_3D:
    rng_3d = np.random.RandomState(42)
    sub_3d = rng_3d.choice(len(valid_raw_vps), MAX_POINTS_3D, replace=False)
else:
    sub_3d = np.arange(len(valid_raw_vps))

ev1_sub = eigenvectors_L[sub_3d, 1]
ev2_sub = eigenvectors_L[sub_3d, 2]
ev3_sub = eigenvectors_L[sub_3d, 3]
vps_sub = valid_raw_vps[sub_3d]

fig_3d = go.Figure()
for vp in fg_sorted_sc:
    m = vps_sub == vp
    if m.sum() == 0:
        continue
    fig_3d.add_trace(go.Scatter3d(
        x=ev1_sub[m], y=ev2_sub[m], z=ev3_sub[m],
        mode='markers',
        marker=dict(size=3, color=fg_colors_sc.get(vp, 'gray'), opacity=0.7),
        name=f'{vp} ({m.sum()})',
        hovertemplate=f'{vp}<br>EV1=%{{x:.4f}}<br>EV2=%{{y:.4f}}<br>EV3=%{{z:.4f}}<extra></extra>'
    ))

fig_3d.update_layout(
    title=f'Laplacian EVs 1-3 (n={len(sub_3d)} subsample)',
    scene=dict(
        xaxis_title='EV1',
        yaxis_title='EV2',
        zaxis_title='EV3',
        aspectmode='cube',
    ),
    width=950, height=750,
    legend=dict(itemsizing='constant'),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig_3d.show()


In [ ]:
# ============================================================
# Angle distribution — histogram with log scale
# ============================================================

theta_vp = np.arctan2(eigenvectors_L[:, EV_Y], eigenvectors_L[:, EV_X])

fg_colors_hist = {
    'right': 'blue', 'backright': 'cornflowerblue', 'frontright': 'deepskyblue',
    'left': 'red', 'backleft': 'salmon', 'frontleft': 'lightcoral',
    'back': 'orange', 'front': 'green',
}
fg_counts_h = Counter(valid_raw_vps)
fg_sorted_h = sorted(fg_counts_h.keys(), key=lambda v: -fg_counts_h[v])
fg_sorted_h = [v for v in fg_sorted_h if fg_counts_h[v] >= 5]

# Stacked histogram — linear and log
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

bins = np.linspace(-np.pi, np.pi, 80)

for ax, log_scale, title in [(axes[0], False, 'Linear scale'), (axes[1], True, 'Log scale')]:
    # Plot each viewpoint as stacked histogram
    bottom = np.zeros(len(bins) - 1)
    for vp in reversed(fg_sorted_h):  # plot smallest on top
        m = valid_raw_vps == vp
        counts, _ = np.histogram(theta_vp[m], bins=bins)
        ax.bar(bins[:-1], counts, width=bins[1]-bins[0], bottom=bottom,
               color=fg_colors_hist.get(vp, 'gray'), alpha=0.7, label=vp,
               edgecolor='none')
        bottom += counts
    
    if log_scale:
        ax.set_yscale('log')
        ax.set_ylim(0.5, None)
    
    ax.set_xlabel(f'atan2({EV_Y_NAME}, {EV_X_NAME}) [radians]')
    ax.set_ylabel('Count')
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=4, loc='upper center')
    ax.set_xlim(-np.pi, np.pi)
    
    # Add degree labels on top axis
    ax2 = ax.twiny()
    ax2.set_xlim(-180, 180)
    ax2.set_xlabel('degrees')

plt.suptitle('Viewpoint angle distribution (fine-grained, stacked)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# --- Circular mean per viewpoint for reference ---
# circular_mean imported from src.utils.circular_stats

print('Circular means:')
for vp in fg_sorted_h:
    m = valid_raw_vps == vp
    cm = circular_mean(theta_vp[m])
    print(f'  {vp:>15}: {np.degrees(cm):+.1f}°')

# AUC
def circular_distance(a, b):
    d = np.abs(a - b)
    return np.minimum(d, 2*np.pi - d)

N_sub_ca = min(3000, len(theta_vp))
rng_ca = np.random.RandomState(42)
sub_ca = rng_ca.choice(len(theta_vp), N_sub_ca, replace=False)
vp_sub_ca = valid_vps[sub_ca]
triu_ca = np.triu_indices(N_sub_ca, k=1)
same_vp_ca = vp_sub_ca[triu_ca[0]] == vp_sub_ca[triu_ca[1]]

d_angle_ca = circular_distance(theta_vp[sub_ca[triu_ca[0]]], theta_vp[sub_ca[triu_ca[1]]])
sim_angle_ca = 1.0 - d_angle_ca / np.pi
auc_angle_ca = roc_auc_score(same_vp_ca, sim_angle_ca)

from scipy.spatial.distance import pdist, squareform
d_4d = squareform(pdist(eigenvectors_L[sub_ca, 1:5], 'euclidean'))
auc_4d_ca = roc_auc_score(same_vp_ca, -d_4d[triu_ca])

print(f'\nViewpoint AUC (circular angle):  {auc_angle_ca:.4f}')
print(f'Viewpoint AUC (4D euclidean):    {auc_4d_ca:.4f}')


In [ ]:
# ============================================================
# Polar plot with eigenvector magnitude as radius
# ============================================================
# theta = atan2(EV2, EV1) = viewpoint angle
# r = sqrt(EV1² + EV2²) = eigenvector magnitude (confidence)

ev1 = eigenvectors_L[:, EV_X]
ev2 = eigenvectors_L[:, EV_Y]
theta_polar = np.arctan2(ev2, ev1)
r_polar = np.sqrt(ev1**2 + ev2**2)

print(f'Radius range: {r_polar.min():.5f} to {r_polar.max():.5f}, mean={r_polar.mean():.5f}')

fg_colors_p = {
    'right': 'blue', 'backright': 'cornflowerblue', 'frontright': 'deepskyblue',
    'left': 'red', 'backleft': 'salmon', 'frontleft': 'lightcoral',
    'back': 'orange', 'front': 'green',
}
fg_counts_p = Counter(valid_raw_vps)
fg_sorted_p = sorted(fg_counts_p.keys(), key=lambda v: -fg_counts_p[v])
fg_sorted_p = [v for v in fg_sorted_p if fg_counts_p[v] >= 5]

# Plot 1: colored by fine-grained viewpoint
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='polar')

for vp in fg_sorted_p:
    m = valid_raw_vps == vp
    ax.scatter(theta_polar[m], r_polar[m],
               c=fg_colors_p.get(vp, 'gray'), s=4, alpha=0.4,
               label=f'{vp} ({m.sum()})', edgecolors='none')

ax.set_title(f'{EV_X_NAME}-{EV_Y_NAME} polar: angle=viewpoint, radius=magnitude', pad=20, fontsize=12)
ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.05, 1), markerscale=3)
plt.tight_layout()
plt.show()

# Plot 2: colored by visibility
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='polar')
sc = ax.scatter(theta_polar, r_polar, c=visibility, s=4, alpha=0.4, cmap='viridis', edgecolors='none')
plt.colorbar(sc, ax=ax, label='Visibility', shrink=0.8)
ax.set_title(f'{EV_X_NAME}-{EV_Y_NAME} polar: colored by visibility', pad=20, fontsize=12)
plt.tight_layout()
plt.show()

# Correlation: radius vs visibility
from scipy.stats import pearsonr
r_vis_corr, _ = pearsonr(r_polar, visibility)
print(f'\nRadius-visibility correlation: r={r_vis_corr:.4f}')
print(f'Radius-{EV_X_NAME} correlation: r={pearsonr(r_polar, np.abs(ev1))[0]:.4f}')


In [ ]:
# ============================================================
# Two-angle viewpoint metric (circular distance)
# ============================================================
# theta1 = atan2(primary EV pair), theta2 = atan2(EV4, EV3)
# Distance = sqrt(circ_dist(θ1)² + circ_dist(θ2)²)

theta1 = np.arctan2(eigenvectors_L[:, EV_Y], eigenvectors_L[:, EV_X])
theta2 = np.arctan2(eigenvectors_L[:, 4], eigenvectors_L[:, 3])

def circ_dist(a, b):
    d = np.abs(a - b)
    return np.minimum(d, 2*np.pi - d)

# Pairwise circular distance matrix
N_sub_2a = min(3000, len(theta1))
rng_2a = np.random.RandomState(42)
sub_2a = rng_2a.choice(len(theta1), N_sub_2a, replace=False)
vp_sub_2a = valid_vps[sub_2a]
triu_2a = np.triu_indices(N_sub_2a, k=1)
same_vp_2a = vp_sub_2a[triu_2a[0]] == vp_sub_2a[triu_2a[1]]

# Two-angle circular distance
d1 = circ_dist(theta1[sub_2a[triu_2a[0]]], theta1[sub_2a[triu_2a[1]]])
d2 = circ_dist(theta2[sub_2a[triu_2a[0]]], theta2[sub_2a[triu_2a[1]]])
d_2angle = np.sqrt(d1**2 + d2**2)
sim_2angle = -d_2angle
auc_2angle = roc_auc_score(same_vp_2a, sim_2angle)

# Comparisons
# Single angle
sim_1angle = -d1
auc_1angle = roc_auc_score(same_vp_2a, sim_1angle)

# 4D euclidean (baseline)
from scipy.spatial.distance import pdist, squareform
d_4d = squareform(pdist(eigenvectors_L[sub_2a, 1:5], 'euclidean'))
auc_4d = roc_auc_score(same_vp_2a, -d_4d[triu_2a])

# 4D cleaned euclidean
d_4d_cl = squareform(pdist(evecs_clean[sub_2a, 1:5], 'euclidean'))
auc_4d_cl = roc_auc_score(same_vp_2a, -d_4d_cl[triu_2a])

# Two-angle with different weightings
print('=== Viewpoint AUC ===')
print(f'  Single angle (θ1 only):        {auc_1angle:.4f}')
print(f'  Two angles (θ1 + θ2):          {auc_2angle:.4f}')
print(f'  4D euclidean (raw):            {auc_4d:.4f}')
print(f'  4D euclidean (vis-cleaned):    {auc_4d_cl:.4f}')

# Try weighted two-angle: emphasize θ1 more
print(f'\n--- Weighted two-angle distance ---')
for w2 in [0.0, 0.25, 0.5, 1.0, 2.0]:
    d_w = np.sqrt(d1**2 + w2 * d2**2)
    auc_w = roc_auc_score(same_vp_2a, -d_w)
    print(f'  w2={w2:.2f}: AUC={auc_w:.4f}')

# --- kNN classification ---
print(f'\n--- kNN classification (seeds=10/vp) ---')

# Custom circular distance for kNN
def two_angle_dist(a, b):
    d1 = circ_dist(a[0], b[0])
    d2 = circ_dist(a[1], b[1])
    return np.sqrt(d1**2 + d2**2)

# Prepare angle features
angles_2d = np.stack([theta1, theta2], axis=1)  # [N, 2]

rng_kn = np.random.RandomState(RANDOM_SEED)
si_kn = []
for vp in unique_vps:
    vp_idx = np.where((valid_vps == vp) & is_pure_vp)[0]
    if len(vp_idx) == 0: vp_idx = np.where(valid_vps == vp)[0]
    si_kn.extend(rng_kn.choice(vp_idx, size=min(10, len(vp_idx)), replace=False).tolist())
tr_2a = np.array(si_kn)
te_2a = np.array([i for i in range(len(theta1)) if i not in set(si_kn)])

# kNN with custom circular metric
clf_2a = KNeighborsClassifier(n_neighbors=5, metric=two_angle_dist)
clf_2a.fit(angles_2d[tr_2a], valid_vps[tr_2a])
pred_2a = clf_2a.predict(angles_2d[te_2a])
acc_2a = (pred_2a == valid_vps[te_2a]).mean()

raw_te_2a = valid_raw_vps[te_2a]
relaxed_2a = np.array([relaxed_correct(raw_te_2a[i], pred_2a[i])
                        for i in range(len(pred_2a))]).mean()

# Baselines
clf_4d = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
clf_4d.fit(eigenvectors_L[tr_2a, 1:5], valid_vps[tr_2a])
pred_4d = clf_4d.predict(eigenvectors_L[te_2a, 1:5])
acc_4d = (pred_4d == valid_vps[te_2a]).mean()
relaxed_4d = np.array([relaxed_correct(raw_te_2a[i], pred_4d[i])
                        for i in range(len(pred_4d))]).mean()

clf_cl = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
clf_cl.fit(evecs_clean[tr_2a, 1:5], valid_vps[tr_2a])
pred_cl = clf_cl.predict(evecs_clean[te_2a, 1:5])
acc_cl = (pred_cl == valid_vps[te_2a]).mean()
relaxed_cl = np.array([relaxed_correct(raw_te_2a[i], pred_cl[i])
                        for i in range(len(pred_cl))]).mean()

print(f'  Two-angle circular:   strict={acc_2a:.4f}, relaxed={relaxed_2a:.4f}')
print(f'  4D euclidean (raw):   strict={acc_4d:.4f}, relaxed={relaxed_4d:.4f}')
print(f'  4D euclidean (clean): strict={acc_cl:.4f}, relaxed={relaxed_cl:.4f}')


In [ ]:
# ============================================================
# All EV pair combinations: which pairing gives best AUC?
# ============================================================
# Test all pairs of (EVi, EVj) for atan2 angle + magnitude.
# Also test all 2-pair combinations for the 4D representation.

from scipy.stats import pearsonr

N_sub_ep = min(3000, len(valid_fvs))
rng_ep = np.random.RandomState(42)
sub_ep = rng_ep.choice(len(valid_fvs), N_sub_ep, replace=False)
vp_sub_ep = valid_vps[sub_ep]
triu_ep = np.triu_indices(N_sub_ep, k=1)
same_vp_ep = vp_sub_ep[triu_ep[0]] == vp_sub_ep[triu_ep[1]]

def circ_dist(a, b):
    d = np.abs(a - b)
    return np.minimum(d, 2*np.pi - d)

# --- Single pair: atan2 angle AUC ---
print('=== Single EV pair: atan2 angle AUC ===')
print(f'{"pair":>8} {"angle_AUC":>10} {"eucl_AUC":>10} {"vis_corr_a":>12} {"vis_corr_b":>12}')

single_results = []
for i in range(1, min(8, eigenvectors_L.shape[1])):
    for j in range(i+1, min(8, eigenvectors_L.shape[1])):
        ev_i = eigenvectors_L[:, i]
        ev_j = eigenvectors_L[:, j]
        
        theta_ij = np.arctan2(ev_j, ev_i)
        
        # Circular angle AUC
        d_ang = circ_dist(theta_ij[sub_ep[triu_ep[0]]], theta_ij[sub_ep[triu_ep[1]]])
        auc_ang = roc_auc_score(same_vp_ep, -d_ang)
        
        # 2D euclidean AUC (angle + magnitude)
        emb_ij = np.stack([ev_i, ev_j], axis=1)
        d_euc = squareform(pdist(emb_ij[sub_ep], 'euclidean'))
        auc_euc = roc_auc_score(same_vp_ep, -d_euc[triu_ep])
        
        # Visibility correlation
        r_i, _ = pearsonr(ev_i, visibility)
        r_j, _ = pearsonr(ev_j, visibility)
        
        single_results.append((i, j, auc_ang, auc_euc, r_i, r_j))
        print(f'  ({i},{j}) {auc_ang:>10.4f} {auc_euc:>10.4f} {r_i:>+10.4f}   {r_j:>+10.4f}')

# --- Two-pair combinations: 4D AUC ---
print(f'\n=== Two EV pairs: 4D euclidean AUC ===')
print(f'{"pairs":>12} {"4D_AUC":>8}')

# Test interesting combinations
pair_combos = [
    ((1,2), (3,4)),  # original
    ((1,2), (2,3)),  # overlapping
    ((1,3), (2,4)),  # cross
    ((1,2), (3,5)),
    ((1,2), (4,5)),
    ((2,3), (4,5)),
    ((1,3), (2,5)),
    ((1,4), (2,3)),
]

for (a, b), (c, d) in pair_combos:
    if max(a,b,c,d) >= eigenvectors_L.shape[1]:
        continue
    emb_4 = eigenvectors_L[sub_ep][:, [a, b, c, d]]
    d_4 = squareform(pdist(emb_4, 'euclidean'))
    auc_4 = roc_auc_score(same_vp_ep, -d_4[triu_ep])
    print(f'  ({a},{b})+({c},{d}) {auc_4:>8.4f}')

# --- What does each EV capture? ---
print(f'\n=== Per-EV viewpoint means ===')
print(f'{"EV":>4} {"vis_r":>8} {"eigenval":>10}', end='')
for vp in ['right', 'left', 'front', 'back']:
    print(f' {vp:>10}', end='')
print()

for i in range(1, min(8, eigenvectors_L.shape[1])):
    r_vis, _ = pearsonr(eigenvectors_L[:, i], visibility)
    print(f'{i:>4} {r_vis:>+8.3f} {eigenvalues_L[i]:>10.5f}', end='')
    for vp in ['right', 'left', 'front', 'back']:
        m = valid_vps == vp
        print(f' {eigenvectors_L[m, i].mean():>+10.5f}', end='')
    print()


In [ ]:
# ============================================================
# EV2-EV3 angle: front/back vs left/right = true azimuth?
# ============================================================

theta_23 = np.arctan2(eigenvectors_L[:, 2], eigenvectors_L[:, 3])
r_23 = np.sqrt(eigenvectors_L[:, 2]**2 + eigenvectors_L[:, 3]**2)

fg_colors_23 = {
    'right': 'blue', 'backright': 'cornflowerblue', 'frontright': 'deepskyblue',
    'left': 'red', 'backleft': 'salmon', 'frontleft': 'lightcoral',
    'back': 'orange', 'front': 'green',
}
fg_counts_23 = Counter(valid_raw_vps)
fg_sorted_23 = sorted(fg_counts_23.keys(), key=lambda v: -fg_counts_23[v])
fg_sorted_23 = [v for v in fg_sorted_23 if fg_counts_23[v] >= 5]

# --- Polar plot with magnitude ---
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='polar')
for vp in fg_sorted_23:
    m = valid_raw_vps == vp
    ax.scatter(theta_23[m], r_23[m], c=fg_colors_23.get(vp, 'gray'),
               s=30, alpha=0.4, label=f'{vp} ({m.sum()})', edgecolors='none')
ax.set_title('atan2(EV2, EV3): front/back vs left/right', pad=20, fontsize=12)
ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.05, 1), markerscale=3)
plt.tight_layout()
plt.show()

# --- Stacked histogram (linear + log) ---
bins = np.linspace(-np.pi, np.pi, 80)
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for ax, log_scale, title in [(axes[0], False, 'Linear'), (axes[1], True, 'Log')]:
    bottom = np.zeros(len(bins) - 1)
    for vp in reversed(fg_sorted_23):
        m = valid_raw_vps == vp
        counts, _ = np.histogram(theta_23[m], bins=bins)
        ax.bar(bins[:-1], counts, width=bins[1]-bins[0], bottom=bottom,
               color=fg_colors_23.get(vp, 'gray'), alpha=0.7, label=vp, edgecolor='none')
        bottom += counts
    if log_scale:
        ax.set_yscale('log')
        ax.set_ylim(0.5, None)
    ax.set_xlabel('atan2(EV2, EV3) [radians]')
    ax.set_ylabel('Count')
    ax.set_title(f'{title} scale')
    ax.legend(fontsize=8, ncol=4, loc='upper center')
    ax2 = ax.twiny()
    ax2.set_xlim(-180, 180)
    ax2.set_xlabel('degrees')

plt.suptitle('atan2(EV2, EV3) distribution (fine-grained)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# --- Per-viewpoint circular means ---
# circular_mean imported from src.utils.circular_stats

print('Circular means for atan2(EV2, EV3):')
for vp in fg_sorted_23:
    m = valid_raw_vps == vp
    cm = circular_mean(theta_23[m])
    print(f'  {vp:>15}: {np.degrees(cm):+.1f}°')

# --- AUC comparison ---
def circ_dist(a, b):
    d = np.abs(a - b)
    return np.minimum(d, 2*np.pi - d)

N_sub_23 = min(3000, len(theta_23))
rng_23 = np.random.RandomState(42)
sub_23 = rng_23.choice(len(theta_23), N_sub_23, replace=False)
vp_sub_23 = valid_vps[sub_23]
triu_23 = np.triu_indices(N_sub_23, k=1)
same_vp_23 = vp_sub_23[triu_23[0]] == vp_sub_23[triu_23[1]]

# atan2(EV2, EV3) angle
d_23 = circ_dist(theta_23[sub_23[triu_23[0]]], theta_23[sub_23[triu_23[1]]])
auc_23_angle = roc_auc_score(same_vp_23, -d_23)

# atan2(EV2, EV1) angle (original)
theta_12 = np.arctan2(eigenvectors_L[:, EV_Y], eigenvectors_L[:, EV_X])
d_12 = circ_dist(theta_12[sub_23[triu_23[0]]], theta_12[sub_23[triu_23[1]]])
auc_12_angle = roc_auc_score(same_vp_23, -d_12)

# EV2,EV3 euclidean
emb_23 = eigenvectors_L[sub_23][:, [2, 3]]
d_23_euc = squareform(pdist(emb_23, 'euclidean'))
auc_23_euc = roc_auc_score(same_vp_23, -d_23_euc[triu_23])

# Visibility correlation
r_vis_23, _ = pearsonr(theta_23, visibility)
r_vis_12, _ = pearsonr(theta_12, visibility)

print(f'\n=== AUC comparison ===')
print(f'  atan2(EV2,EV3) angle:  {auc_23_angle:.4f}  (vis_corr={r_vis_23:+.4f})')
print(f'  atan2({EV_Y_NAME},{EV_X_NAME}) angle:  {auc_12_angle:.4f}  (vis_corr={r_vis_12:+.4f})')
print(f'  (EV2,EV3) euclidean:   {auc_23_euc:.4f}')


In [ ]:
# ============================================================
# Edge proximity: how close is the detection to the image edge?
# ============================================================
# Detections cut by the frame should have bbox touching the edge.
# Metric: min distance from bbox to any image edge, normalized
# by image size. 0 = touching edge, 1 = centered.

# Get image dimensions from COCO (no image loading needed)
coco_images = coco_loader.images  # dict: uuid -> COCOImage
det_to_image_uuid = {m.detection_id: m.gt_annotation.image_uuid for m in matched}

print('Computing edge proximity...')
t0_ep = time.time()

edge_proximity = np.zeros(len(valid_dets), dtype=np.float32)

# Pre-load all bboxes from batch files efficiently
from src.utils.batch_storage import load_batch_from_file
det_bboxes = {}
batch_files = sorted(dataset._index['batch_to_detections'].keys())
for bf in batch_files:
    batch_data = load_batch_from_file(dataset.output_root / bf)
    for did, det in batch_data.items():
        if did in set(valid_dets):
            det_bboxes[did] = det.bbox.int().tolist()
    del batch_data

print(f'  Loaded {len(det_bboxes)} bboxes in {time.time()-t0_ep:.1f}s')

for i, det_id in enumerate(valid_dets):
    img_uuid = det_to_image_uuid.get(det_id)
    if img_uuid is None or img_uuid not in coco_images:
        edge_proximity[i] = 1.0
        continue
    
    coco_img = coco_images[img_uuid]
    img_w, img_h = coco_img.width, coco_img.height
    
    bbox = det_bboxes.get(det_id)
    if bbox is None:
        edge_proximity[i] = 1.0
        continue
    sx1, sy1, sx2, sy2 = bbox
    
    d_left = max(sx1, 0)
    d_top = max(sy1, 0)
    d_right = max(img_w - sx2, 0)
    d_bottom = max(img_h - sy2, 0)
    
    min_edge_dist = min(d_left, d_top, d_right, d_bottom)
    diag = np.sqrt(img_w**2 + img_h**2)
    edge_proximity[i] = min_edge_dist / diag

print(f'  Done in {time.time()-t0_ep:.1f}s')
print(f'Edge proximity: min={edge_proximity.min():.4f}, max={edge_proximity.max():.4f}, '
      f'mean={edge_proximity.mean():.4f}')
print(f'Touching edge (< 0.01): {(edge_proximity < 0.01).sum()}/{len(edge_proximity)}')

# --- Polar plot colored by edge proximity ---
ev1_ep = eigenvectors_L[:, EV_X]
ev2_ep = eigenvectors_L[:, EV_Y]
theta_ep = np.arctan2(ev2_ep, ev1_ep)
r_ep = np.sqrt(ev1_ep**2 + ev2_ep**2)

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='polar')
sc = ax.scatter(theta_ep, r_ep, c=edge_proximity, s=4, alpha=0.4, cmap='RdYlGn', edgecolors='none')
plt.colorbar(sc, ax=ax, label='Edge proximity (0=touching, 1=centered)', shrink=0.8)
ax.set_title(f'{EV_X_NAME}-{EV_Y_NAME} polar: colored by edge proximity', pad=20, fontsize=12)
ax.set_rticks([])
plt.tight_layout()
plt.show()

# --- Correlation with visibility and viewpoint angle ---
from scipy.stats import pearsonr
r_vis_edge, _ = pearsonr(edge_proximity, visibility)
r_angle_edge, _ = pearsonr(edge_proximity, theta_ep)
print(f'\nCorrelations:')
print(f'  Edge proximity vs visibility: r={r_vis_edge:+.4f}')
print(f'  Edge proximity vs angle:      r={r_angle_edge:+.4f}')

# --- Error rate by edge proximity ---
# Use pred_report from cell 6 (laplacian_4d_eval)
ep_test = edge_proximity[test_best]
hard_ep = ((pred_report != valid_vps[test_best]) &
           ~np.array([relaxed_correct(valid_raw_vps[test_best][i], pred_report[i])
                      for i in range(len(pred_report))]))

print(f'\nError rate by edge proximity:')
for lo, hi in [(0, 0.01), (0.01, 0.05), (0.05, 0.1), (0.1, 0.2), (0.2, 1.01)]:
    band = (ep_test >= lo) & (ep_test < hi)
    if band.sum() < 10:
        continue
    err = hard_ep[band].mean()
    print(f'  [{lo:.2f}-{hi:.2f}): n={band.sum():>5}, hard_err={err:.3f}')


In [ ]:
# ============================================================
# Laplacian 3D distance histograms: same vs different viewpoint
# ============================================================

from scipy.spatial.distance import pdist, squareform

emb_hist = eigenvectors_L[:, 1:4]

N_sub_h = min(3000, len(valid_fvs))
rng_h = np.random.RandomState(42)
sub_h = rng_h.choice(len(valid_fvs), N_sub_h, replace=False)

vp_sub_h = valid_vps[sub_h]
raw_vp_sub_h = valid_raw_vps[sub_h]
vis_sub_h = visibility[sub_h]
ep_sub_h = edge_proximity[sub_h]

triu_h = np.triu_indices(N_sub_h, k=1)

d_h = squareform(pdist(emb_hist[sub_h], 'euclidean'))
dists_flat = d_h[triu_h]

# Same coarse viewpoint
same_coarse = vp_sub_h[triu_h[0]] == vp_sub_h[triu_h[1]]

# Relaxed same: share any component (using src.evaluation.viewpoint_accuracy)
same_relaxed = np.array([relaxed_same_viewpoint(raw_vp_sub_h[i], raw_vp_sub_h[j])
                          for i, j in zip(triu_h[0], triu_h[1])])

# Split by edge proximity
both_edge = (ep_sub_h[triu_h[0]] < 0.01) & (ep_sub_h[triu_h[1]] < 0.01)
neither_edge = (ep_sub_h[triu_h[0]] >= 0.01) & (ep_sub_h[triu_h[1]] >= 0.01)
one_edge = ~both_edge & ~neither_edge

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: overall same vs diff
ax = axes[0, 0]
ax.hist(dists_flat[same_relaxed], bins=80, alpha=0.5, density=True, color='green', label=f'Same VP (relaxed, n={same_relaxed.sum()})')
ax.hist(dists_flat[~same_relaxed], bins=80, alpha=0.5, density=True, color='red', label=f'Diff VP (n={(~same_relaxed).sum()})')
ax.set_xlabel('Laplacian 3D distance')
ax.set_ylabel('Density')
ax.set_title('All pairs')
ax.legend(fontsize=8)

# Plot 2: neither near edge
ax = axes[0, 1]
mask = neither_edge & same_relaxed
ax.hist(dists_flat[mask], bins=80, alpha=0.5, density=True, color='green', label=f'Same VP (n={mask.sum()})')
mask = neither_edge & ~same_relaxed
ax.hist(dists_flat[mask], bins=80, alpha=0.5, density=True, color='red', label=f'Diff VP (n={mask.sum()})')
ax.set_xlabel('Laplacian 3D distance')
ax.set_title('Neither near edge (ep >= 0.01)')
ax.legend(fontsize=8)

# Plot 3: both near edge
ax = axes[1, 0]
mask = both_edge & same_relaxed
ax.hist(dists_flat[mask], bins=80, alpha=0.5, density=True, color='green', label=f'Same VP (n={mask.sum()})')
mask = both_edge & ~same_relaxed
ax.hist(dists_flat[mask], bins=80, alpha=0.5, density=True, color='red', label=f'Diff VP (n={mask.sum()})')
ax.set_xlabel('Laplacian 3D distance')
ax.set_title('Both near edge (ep < 0.01)')
ax.legend(fontsize=8)

# Plot 4: one near edge, one not
ax = axes[1, 1]
mask = one_edge & same_relaxed
ax.hist(dists_flat[mask], bins=80, alpha=0.5, density=True, color='green', label=f'Same VP (n={mask.sum()})')
mask = one_edge & ~same_relaxed
ax.hist(dists_flat[mask], bins=80, alpha=0.5, density=True, color='red', label=f'Diff VP (n={mask.sum()})')
ax.set_xlabel('Laplacian 3D distance')
ax.set_title('One near edge, one not')
ax.legend(fontsize=8)

plt.suptitle('Laplacian 3D distance: same vs different viewpoint', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Per-group AUC
for name, mask_group in [('All', np.ones(len(dists_flat), dtype=bool)),
                          ('Neither edge', neither_edge),
                          ('Both edge', both_edge),
                          ('One edge', one_edge)]:
    if mask_group.sum() < 100:
        continue
    auc = roc_auc_score(same_relaxed[mask_group], -dists_flat[mask_group])
    print(f'  {name:>15}: AUC={auc:.4f} (n={mask_group.sum()})')


In [ ]:
# ============================================================
# Full distance distribution
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax = axes[0]
ax.hist(dists_flat, bins=200, density=True, color='blue', alpha=0.5)
ax.set_xlabel('Laplacian 3D distance')
ax.set_ylabel('Density')
ax.set_title('Full distance distribution (linear)')

# Log scale
ax = axes[1]
ax.hist(dists_flat, bins=200, density=True, color='blue', alpha=0.5)
ax.set_yscale('log')
ax.set_xlabel('Laplacian 3D distance')
ax.set_ylabel('Density (log)')
ax.set_title('Full distance distribution (log)')

plt.tight_layout()
plt.show()

# Stats
print(f'Distance stats:')
print(f'  min={dists_flat.min():.6f}, max={dists_flat.max():.6f}')
print(f'  mean={dists_flat.mean():.6f}, median={np.median(dists_flat):.6f}')
print(f'  std={dists_flat.std():.6f}')
print(f'  Percentiles: 1%={np.percentile(dists_flat,1):.6f}, 99%={np.percentile(dists_flat,99):.6f}')
print(f'  > 0.2: {(dists_flat > 0.2).sum()} pairs ({(dists_flat > 0.2).mean()*100:.2f}%)')
print(f'  > 0.3: {(dists_flat > 0.3).sum()} pairs ({(dists_flat > 0.3).mean()*100:.2f}%)')

# What are the 0.32 island pairs?
island_mask = dists_flat > 0.25
island_i = triu_h[0][island_mask]
island_j = triu_h[1][island_mask]
print(f'\nIsland pairs (d > 0.25): {island_mask.sum()}')
if island_mask.sum() > 0 and island_mask.sum() < 200:
    # What are these detections?
    island_det_ids = set()
    for ii, jj in zip(island_i[:20], island_j[:20]):
        island_det_ids.add(sub_h[ii])
        island_det_ids.add(sub_h[jj])
    print(f'  Unique detections in island: {len(island_det_ids)}')
    print(f'  Their viewpoints: {Counter(valid_vps[list(island_det_ids)])}')
    print(f'  Their visibility: {visibility[list(island_det_ids)].mean():.3f}')
    print(f'  Their edge proximity: {edge_proximity[list(island_det_ids)].mean():.4f}')
    # Are they a few outlier detections paired with everything?
    all_island_dets = np.concatenate([sub_h[island_i], sub_h[island_j]])
    det_counts = Counter(all_island_dets)
    print(f'  Most common detections in island pairs:')
    for det_idx, count in det_counts.most_common(5):
        print(f'    idx={det_idx}: {count} pairs, vp={valid_vps[det_idx]}, vis={visibility[det_idx]:.3f}, ep={edge_proximity[det_idx]:.4f}')


In [ ]:
# ============================================================
# Pair inspector: view detection pairs by distance range
# ============================================================
# Specify distance range and whether same/diff viewpoint.
# Re-run cell to see different random pairs.

DIST_MIN = 0.05      # minimum Laplacian 3D distance
DIST_MAX = 0.06      # maximum
PAIR_TYPE = 'same'   # 'same' (relaxed same VP) or 'diff' or 'any'

from PIL import Image as PILImage

# Filter pairs
dist_mask = (dists_flat >= DIST_MIN) & (dists_flat <= DIST_MAX)
if PAIR_TYPE == 'same':
    pair_mask = dist_mask & same_relaxed
elif PAIR_TYPE == 'diff':
    pair_mask = dist_mask & ~same_relaxed
else:
    pair_mask = dist_mask

n_pairs = pair_mask.sum()
print(f'Pairs with d in [{DIST_MIN}, {DIST_MAX}], type={PAIR_TYPE}: {n_pairs}')

if n_pairs == 0:
    print('No pairs found!')
else:
    # Random pair
    pair_indices = np.where(pair_mask)[0]
    pick = np.random.choice(pair_indices)
    i_sub, j_sub = triu_h[0][pick], triu_h[1][pick]
    i_global, j_global = sub_h[i_sub], sub_h[j_sub]
    
    dist_val = dists_flat[pick]
    
    # Get info for both detections
    for label, gi in [('Detection A', i_global), ('Detection B', j_global)]:
        did = valid_dets[gi]
        print(f'\n{label}: {did[:40]}')
        print(f'  Raw VP: {valid_raw_vps[gi]}, Coarse VP: {valid_vps[gi]}')
        print(f'  Visibility: {visibility[gi]:.3f}, Edge prox: {edge_proximity[gi]:.4f}')
        print(f'  Angle: {np.degrees(np.arctan2(eigenvectors_L[gi, EV_Y], eigenvectors_L[gi, EV_X])):.1f}°')
        print(f'  {EV_X_NAME}-{EV_Y_NAME}: [{eigenvectors_L[gi, EV_X]:.5f}, {eigenvectors_L[gi, EV_Y]:.5f}]  EV3={eigenvectors_L[gi, 3]:.5f}  EV4={eigenvectors_L[gi, 4]:.5f}')
    
    print(f'\nLaplacian 3D distance: {dist_val:.6f}')
    print(f'Same VP (relaxed): {same_relaxed[pick]}')
    
    # Show images
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    for ax, gi, label in [(axes[0], i_global, 'A'), (axes[1], j_global, 'B')]:
        did = valid_dets[gi]
        det_obj = dataset.get_detection(did)
        img = PILImage.open(det_obj.image_path).convert('RGB')
        img_w, img_h = img.size
        sx1, sy1, sx2, sy2 = det_obj.bbox.int().tolist()
        sx1, sy1 = max(0, sx1), max(0, sy1)
        sx2, sy2 = min(img_w, sx2), min(img_h, sy2)
        crop = img.crop((sx1, sy1, sx2, sy2))
        ax.imshow(crop)
        ax.set_title(
            f'{label}: {valid_raw_vps[gi]} (coarse: {valid_vps[gi]})\n'
            f'vis={visibility[gi]:.2f}, edge={edge_proximity[gi]:.3f}\n'
            f'angle={np.degrees(np.arctan2(eigenvectors_L[gi, EV_Y], eigenvectors_L[gi, EV_X])):.1f}°',
            fontsize=10
        )
        ax.axis('off')
    
    fig.suptitle(f'd={dist_val:.4f}, {"SAME" if same_relaxed[pick] else "DIFF"} VP ({PAIR_TYPE})',
                 fontsize=13, y=1.02,
                 color='green' if same_relaxed[pick] else 'red')
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# Investigate the outlier island at d ~ 0.30
# ============================================================
# Hypothesis: the island is caused by a nearly-disconnected
# component of the kNN graph. Points in such a component get
# extreme eigenvector values regardless of actual FV similarity.

from scipy.sparse.csgraph import connected_components

ISLAND_DIST_MIN = 0.15  # anything above this is "island" territory

# --- Step 1: find island pairs in the sampled subset ---
island_mask = dists_flat > ISLAND_DIST_MIN
n_island_pairs = island_mask.sum()
print(f'Island pairs (d > {ISLAND_DIST_MIN}): {n_island_pairs} / {len(dists_flat)} '
      f'({100*n_island_pairs/len(dists_flat):.3f}%)')

if n_island_pairs > 0:
    # Which detections in the subset participate in island pairs?
    i_sub_isl = triu_h[0][island_mask]
    j_sub_isl = triu_h[1][island_mask]
    island_sub_idx = np.unique(np.concatenate([i_sub_isl, j_sub_isl]))
    island_global_idx = sub_h[island_sub_idx]

    print(f'Unique detections on island (in subset): {len(island_sub_idx)} / {len(sub_h)}')

    # --- Step 2: connected components of the full kNN graph ---
    n_comp, comp_labels = connected_components(A, directed=False)
    comp_sizes = np.bincount(comp_labels)
    print(f'\nkNN graph: {n_comp} connected components')
    print(f'  Largest: {comp_sizes.max()} ({100*comp_sizes.max()/len(comp_labels):.2f}%)')
    print(f'  Top 10 sizes: {sorted(comp_sizes, reverse=True)[:10]}')

    # Which components do island detections belong to?
    island_comps = comp_labels[island_global_idx]
    from collections import Counter
    island_comp_counts = Counter(island_comps)
    print(f'\nIsland detections by component:')
    for comp_id, count in island_comp_counts.most_common(10):
        print(f'  Comp {comp_id}: {count} island dets, comp size = {comp_sizes[comp_id]}')

    # --- Step 3: eigenvector values on the island ---
    print('\nEigenvector stats (EVs 1-4):')
    print(f'  {"EV":>4} {"all min":>12} {"all max":>12} {"island min":>14} {"island max":>14}')
    for ev_i in range(1, 5):
        all_vals = eigenvectors_L[:, ev_i]
        isl_vals = eigenvectors_L[island_global_idx, ev_i]
        print(f'  {ev_i:>4} {all_vals.min():>12.5f} {all_vals.max():>12.5f} '
              f'{isl_vals.min():>14.5f} {isl_vals.max():>14.5f}')

    # How many detections are extreme outliers in ANY of EVs 1-4?
    ev14 = eigenvectors_L[:, 1:5]
    # Robust z-score via MAD
    med = np.median(ev14, axis=0)
    mad = np.median(np.abs(ev14 - med), axis=0) + 1e-12
    z = np.abs(ev14 - med) / mad
    extreme_mask = (z > 20).any(axis=1)  # z>20 on any EV
    print(f'\nDetections with |MAD-z| > 20 on any of EVs 1-4: {extreme_mask.sum()} / {len(eigenvectors_L)}')
    print(f'  Of those, on island (subset): {np.isin(island_global_idx, np.where(extreme_mask)[0]).sum()}')

    # --- Step 4: node degrees of island detections ---
    deg_all = degrees
    deg_island = deg_all[island_global_idx]
    print(f'\nNode degree in kNN graph:')
    print(f'  All: median={np.median(deg_all):.1f}, min={deg_all.min()}, max={deg_all.max()}')
    print(f'  Island: median={np.median(deg_island):.1f}, min={deg_island.min()}, max={deg_island.max()}')


In [ ]:
# ============================================================
# Which eigenvector creates the island?
# ============================================================
# Decompose island pair distances per-EV and check each EV's
# distribution for bimodality (bottleneck signature).

emb4 = eigenvectors_L[:, 1:5]  # [N, 4]

# Per-EV contribution to squared distance for each sampled pair
diffs_sq = np.zeros((len(triu_h[0]), 4))
for ev_i in range(4):
    col = emb4[sub_h, ev_i]
    diffs_sq[:, ev_i] = (col[triu_h[0]] - col[triu_h[1]]) ** 2
total_sq = diffs_sq.sum(axis=1)
contrib_frac = diffs_sq / (total_sq[:, None] + 1e-20)  # fraction of squared dist from each EV

print('Mean per-EV contribution to squared distance:')
for name, mask_ in [('All pairs', np.ones(len(total_sq), bool)),
                    ('Island (d>0.15)', dists_flat > 0.15)]:
    if mask_.sum() == 0: continue
    fracs = contrib_frac[mask_].mean(axis=0)
    print(f'  {name:>18}: EV1={fracs[0]:.3f}  EV2={fracs[1]:.3f}  EV3={fracs[2]:.3f}  EV4={fracs[3]:.3f}')

# Histograms of each EV
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

# Top row: full distribution
for ev_i in range(4):
    ax = axes[0, ev_i]
    vals = eigenvectors_L[:, ev_i + 1]
    ax.hist(vals, bins=100, color='steelblue')
    ax.set_title(f'EV{ev_i+1}: full range\nmin={vals.min():.4f}, max={vals.max():.4f}')
    ax.set_xlabel(f'EV{ev_i+1}')
    ax.set_yscale('log')

# Bottom row: log-count, zoomed to show tails
for ev_i in range(4):
    ax = axes[1, ev_i]
    vals = eigenvectors_L[:, ev_i + 1]
    med = np.median(vals)
    mad = np.median(np.abs(vals - med)) + 1e-12
    z = (vals - med) / mad
    ax.hist(z, bins=100, color='darkorange')
    n_outlier = (np.abs(z) > 20).sum()
    ax.set_title(f'EV{ev_i+1}: MAD-z\n|z|>20: {n_outlier} dets')
    ax.set_xlabel(f'EV{ev_i+1} MAD z-score')
    ax.set_yscale('log')
    ax.axvline(-20, color='red', ls='--', alpha=0.5)
    ax.axvline(20, color='red', ls='--', alpha=0.5)

plt.suptitle('Eigenvector distributions — looking for bottleneck signatures', fontsize=13)
plt.tight_layout()
plt.show()

# How many detections have extreme EV4 (the main suspect)
ev4 = eigenvectors_L[:, 4]
thr = -0.1  # clearly in the negative tail based on min=-0.315, others near 0
extreme_ev4 = ev4 < thr
print(f'\nDetections with EV4 < {thr}: {extreme_ev4.sum()} / {len(ev4)}')
print(f'EV4 distribution: 5%={np.percentile(ev4, 5):.5f}, '
      f'50%={np.percentile(ev4, 50):.5f}, '
      f'95%={np.percentile(ev4, 95):.5f}, '
      f'99%={np.percentile(ev4, 99):.5f}')

# What viewpoints do extreme-EV4 detections have?
if extreme_ev4.sum() > 0:
    from collections import Counter
    vp_extreme = Counter(valid_raw_vps[extreme_ev4])
    vp_all = Counter(valid_raw_vps)
    print(f'\nExtreme EV4 viewpoint breakdown:')
    for vp, c in vp_extreme.most_common(10):
        frac_extreme = c / extreme_ev4.sum()
        frac_all = vp_all[vp] / len(valid_raw_vps)
        enrichment = frac_extreme / frac_all if frac_all > 0 else 0
        print(f'  {vp:>15}: {c:4d} ({100*frac_extreme:5.1f}%)  enrichment={enrichment:.2f}x')

    # Visibility comparison
    print(f'\nVisibility of extreme-EV4 dets:')
    print(f'  mean={visibility[extreme_ev4].mean():.3f}, '
          f'median={np.median(visibility[extreme_ev4]):.3f}')
    print(f'Visibility of normal dets:')
    print(f'  mean={visibility[~extreme_ev4].mean():.3f}, '
          f'median={np.median(visibility[~extreme_ev4]):.3f}')


In [ ]:
# ============================================================
# Visualize the 10 extreme-EV4 detections (the bottleneck cluster)
# ============================================================
from PIL import Image as PILImage

ev4 = eigenvectors_L[:, 4]
extreme_idx = np.where(ev4 < -0.1)[0]
# Sort by EV4 value (most extreme first)
extreme_idx = extreme_idx[np.argsort(ev4[extreme_idx])]

print(f'Showing {len(extreme_idx)} extreme-EV4 detections')
print(f'\n{"idx":>6} {"det_id":>20} {"EV4":>10} {"vis":>6} {"edge":>7} {"raw_vp":>15} {"image_uuid":>20}')

det_to_annot = {m.detection_id: m.gt_annotation for m in matched}

n = len(extreme_idx)
ncols = 5
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
axes = np.atleast_2d(axes).ravel()

for plot_i, gi in enumerate(extreme_idx):
    did = valid_dets[gi]
    annot = det_to_annot.get(did)
    img_uuid = annot.image_uuid if annot else "?"
    ind_id = annot.individual_id if annot else "?"
    print(f'  {gi:>6} {did[:18]:>20} {ev4[gi]:>10.5f} {visibility[gi]:>6.3f} '
          f'{edge_proximity[gi]:>7.4f} {valid_raw_vps[gi]:>15} {img_uuid[:18]:>20}  ind={ind_id}')

    det_obj = dataset.get_detection(did)
    img = PILImage.open(det_obj.image_path).convert('RGB')
    iw, ih = img.size
    sx1, sy1, sx2, sy2 = det_obj.bbox.int().tolist()
    sx1, sy1 = max(0, sx1), max(0, sy1)
    sx2, sy2 = min(iw, sx2), min(ih, sy2)
    crop = img.crop((sx1, sy1, sx2, sy2))
    ax = axes[plot_i]
    ax.imshow(crop)
    ax.set_title(f'EV4={ev4[gi]:.3f}, vis={visibility[gi]:.2f}\n'
                 f'{valid_raw_vps[gi]}, ind={str(ind_id)[:10]}',
                 fontsize=9)
    ax.axis('off')

for k in range(n, len(axes)):
    axes[k].axis('off')

plt.suptitle('The 10 extreme-EV4 detections (bottleneck cluster)', fontsize=13)
plt.tight_layout()
plt.show()

# Identity breakdown
print('\nIdentity distribution of extreme-EV4 dets:')
from collections import Counter
inds = [det_to_annot[valid_dets[gi]].individual_id if valid_dets[gi] in det_to_annot else None
        for gi in extreme_idx]
print(f'  Unique individuals: {len(set(inds))}')
print(f'  Counts: {Counter(inds).most_common()}')

# Image breakdown
imgs = [det_to_annot[valid_dets[gi]].image_uuid if valid_dets[gi] in det_to_annot else None
        for gi in extreme_idx]
print(f'\nImage distribution:')
print(f'  Unique images: {len(set(imgs))}')


In [ ]:
# ============================================================
# Does dropping EV4 actually improve viewpoint results?
# ============================================================
# Compare AUC + kNN accuracy for different EV subsets.
# Same train/test split as the 4D baseline cell for a fair comparison.

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

# --- Pairwise AUC on the 3000-sample subset ---
configs = [
    ('EVs 1-2',    [1, 2]),
    ('EVs 1-3',    [1, 2, 3]),
    ('EVs 1,2,3  (drop EV4)', [1, 2, 3]),
    ('EVs 1-4  (baseline)',   [1, 2, 3, 4]),
    ('EVs 1,2,3,5',           [1, 2, 3, 5]),
    ('EVs 1-5',               [1, 2, 3, 4, 5]),
]

print('=== AUC (same-VP pair discrimination) ===')
print(f'{"config":>28} {"AUC":>8}')
for name, evs in configs:
    emb = eigenvectors_L[:, evs]
    d = squareform(pdist(emb[sub_ev], "euclidean"))
    sim = -d[triu_ev]
    auc = roc_auc_score(same_vp_ev, sim)
    print(f'  {name:>28} {auc:>8.4f}')

# --- kNN classification with the EXACT same train/test split as baseline ---
print('\n=== kNN strict / relaxed accuracy (same seeds as baseline) ===')
print(f'{"config":>28} {"strict":>8} {"relaxed":>9}')

raw_test_bb = valid_raw_vps[test_best]

for name, evs in configs:
    emb = eigenvectors_L[:, evs]
    clf = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
    clf.fit(emb[train_best], valid_vps[train_best])
    pred_cfg = clf.predict(emb[test_best])
    strict = (pred_cfg == valid_vps[test_best]).mean()
    relaxed_arr = np.array([
        relaxed_correct(raw_test_bb[i], pred_cfg[i])
        for i in range(len(pred_cfg))
    ])
    relaxed = relaxed_arr.mean()
    print(f'  {name:>28} {strict:>8.4f} {relaxed:>9.4f}')

# --- Island check: is the d~0.30 island gone after dropping EV4? ---
print('\n=== Island pairs (d > 0.15) after dropping EV4 ===')
emb_no4 = eigenvectors_L[:, [1, 2, 3]]
d_no4 = squareform(pdist(emb_no4[sub_h], "euclidean"))[triu_h]
n_island_no4 = (d_no4 > 0.15).sum()
print(f'  EVs 1-4 baseline : {(dists_flat > 0.15).sum():5d} island pairs')
print(f'  EVs 1,2,3 (no 4) : {n_island_no4:5d} island pairs')
print(f'  max dist (1,2,3) : {d_no4.max():.4f}')
print(f'  max dist (1-4)   : {dists_flat.max():.4f}')


# Summary

## Goal
Recover viewpoint from Fisher vectors **without** viewpoint supervision, and
build a continuous same/different-viewpoint distance metric.

## Method
1. Build a **kNN graph** (k=10) on L2-normalized, square-root-normalized FVs.
2. Compute the **normalized graph Laplacian eigenvectors**.
3. Use **EVs 1-3** as the viewpoint-aware embedding.
4. Distance metric: Euclidean distance in the 3D Laplacian embedding.
5. Angle metric: `atan2(EV2, EV1)` and `atan2(EV3, EV2)` give a 2D angular space on which all 8 fine-grained viewpoints separate cleanly.

## Key findings

### EVs 1-3 are the clean viewpoint subspace
| Metric | Value |
|---|---|
| AUC (same-VP discrimination) | **0.8272** |
| kNN strict accuracy (10 seeds/VP) | **74.3%** |
| kNN relaxed accuracy | **80.0%** |

### Higher eigenvectors are nuisance-hijacked
- **EV1**: 84% correlated with **visibility** (fraction of active GMM components). Regressing out visibility recovers some accuracy.
- **EV4**: Acts as a Fiedler vector separating a small weakly-connected subgraph — **~10 near-duplicate, high-visibility, right-profile detections** from a single photo session. Including EV4 creates a spurious "island" at d~0.30 in the distance distribution and *slightly hurts* AUC.
- **EV5+**: Also nuisance. Adding them drops AUC below 0.80.

### Diagnostic pattern
Whenever the kNN graph has a narrow cut (duplicate images, shared lighting, shared visibility pattern), a top Laplacian eigenvector gets hijacked to encode that cut rather than viewpoint. Detection: **bimodal/heavy-tailed eigenvector distribution on a log-scale histogram**.

### Fundamental limit
Low-visibility detections (vis < 0.3) cluster by shared sparsity rather than viewpoint — AUC ~0.46 on this subset regardless of method.


In [ ]:
# ============================================================
# Final metrics: Laplacian 3D (EVs 1-3) kNN classification
# ============================================================

emb_final = eigenvectors_L[:, 1:4]

# kNN classification on the same train/test split as the baseline cell
clf_final = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
clf_final.fit(emb_final[train_best], valid_vps[train_best])
pred_final = clf_final.predict(emb_final[test_best])

strict_final = (pred_final == valid_vps[test_best]).mean()
raw_te = valid_raw_vps[test_best]
relaxed_arr = np.array([relaxed_correct(raw_te[i], pred_final[i])
                         for i in range(len(pred_final))])
relaxed_final = relaxed_arr.mean()

# AUC on the 3000-sample subset (same sub_ev as baseline cell)
d_final = squareform(pdist(emb_final[sub_ev], 'euclidean'))
auc_final = roc_auc_score(same_vp_ev, -d_final[triu_ev])

# Baseline for comparison (original FV cosine)
d_fv = (valid_fvs[sub_ev] @ valid_fvs[sub_ev].T)
auc_fv_baseline = roc_auc_score(same_vp_ev, d_fv[triu_ev])

# --- Headline metrics ---
print(f'{"Metric":>35} {"Value":>10}')
print('-' * 48)
print(f'{"AUC: FV cosine (baseline)":>35} {auc_fv_baseline:>10.4f}')
print(f'{"AUC: Laplacian 3D euclidean":>35} {auc_final:>10.4f}')
print(f'{"kNN strict accuracy":>35} {strict_final:>10.4f}')
print(f'{"kNN relaxed accuracy":>35} {relaxed_final:>10.4f}')
print(f'{"Train / test split":>35} {f"{len(train_best)} / {len(test_best)}":>10}')

# --- Full sklearn classification report (coarse viewpoints) ---
print('\n=== Classification report — coarse viewpoints (strict) ===')
print(classification_report(
    valid_vps[test_best], pred_final,
    labels=list(unique_vps), target_names=list(unique_vps),
    digits=4, zero_division=0,
))

# --- Per-viewpoint strict vs relaxed table ---
print(f'{"viewpoint":>12} {"n_test":>8} {"strict":>10} {"relaxed":>10}')
print('-' * 44)
for vp in unique_vps:
    mask_vp = valid_vps[test_best] == vp
    if mask_vp.sum() == 0:
        continue
    s = (pred_final[mask_vp] == vp).mean()
    r = relaxed_arr[mask_vp].mean()
    print(f'{vp:>12} {mask_vp.sum():>8d} {s:>10.4f} {r:>10.4f}')

# --- Fine-grained (raw) viewpoint breakdown: what did each raw VP get classified as? ---
print('\n=== Raw (fine-grained) viewpoint -> predicted coarse ===')
raw_test_vps_unique = sorted(set(raw_te))
print(f'{"raw_vp":>14} {"n":>6} {"most common pred":>22} {"strict acc":>12} {"relaxed acc":>12}')
print('-' * 70)
for raw_vp in raw_test_vps_unique:
    m = raw_te == raw_vp
    if m.sum() < 3:
        continue
    preds_m = pred_final[m]
    top_pred = Counter(preds_m).most_common(1)[0]
    strict_rv = (preds_m == valid_vps[test_best][m]).mean()
    relaxed_rv = relaxed_arr[m].mean()
    print(f'{raw_vp:>14} {m.sum():>6d} {f"{top_pred[0]} ({top_pred[1]})":>22} '
          f'{strict_rv:>12.4f} {relaxed_rv:>12.4f}')

# --- Confusion matrix plot ---
cm_final = confusion_matrix(valid_vps[test_best], pred_final, labels=list(unique_vps))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm_final, display_labels=list(unique_vps)).plot(
    ax=ax, cmap='Blues', values_format='d', colorbar=True,
)
ax.set_title(f'Confusion matrix — Laplacian 3D kNN\n'
             f'strict={strict_final:.3f}, relaxed={relaxed_final:.3f}',
             fontsize=12)
plt.tight_layout()
plt.show()

# --- Relaxed classification report + confusion matrix ---
# Construct a relaxed ground truth: if the prediction is relaxed-correct
# for the raw VP (e.g. frontright -> front OR right), treat the label as
# the predicted class. Otherwise keep the original coarse label.
y_true_relaxed = np.where(relaxed_arr, pred_final, valid_vps[test_best])

print('\n=== Classification report — coarse viewpoints (relaxed) ===')
print(classification_report(
    y_true_relaxed, pred_final,
    labels=list(unique_vps), target_names=list(unique_vps),
    digits=4, zero_division=0,
))

cm_relaxed = confusion_matrix(y_true_relaxed, pred_final, labels=list(unique_vps))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm_relaxed, display_labels=list(unique_vps)).plot(
    ax=ax, cmap='Greens', values_format='d', colorbar=True,
)
ax.set_title(f'Confusion matrix (relaxed) — Laplacian 3D kNN\n'
             f'relaxed accuracy={relaxed_final:.3f}',
             fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Money shot: 2D angular viewpoint space (all detections)
# ============================================================
# The two Laplacian angles recover the full 8-viewpoint structure
# without any viewpoint supervision.

fig, ax = plt.subplots(figsize=(13, 11))
for vp in fg_sorted_sc:
    m = valid_raw_vps == vp
    ax.scatter(theta1_full[m], theta2_full[m],
               c=fg_colors_sc.get(vp, 'gray'), s=28, alpha=0.6,
               label=f'{vp} ({m.sum()})', edgecolors='none')

# Overlay circular means as X markers
for vp in fg_sorted_sc:
    m = valid_raw_vps == vp
    t1_mean_rad = np.arctan2(np.sin(np.radians(theta1_full[m])).mean(),
                              np.cos(np.radians(theta1_full[m])).mean())
    t2_mean_rad = np.arctan2(np.sin(np.radians(theta2_full[m])).mean(),
                              np.cos(np.radians(theta2_full[m])).mean())
    ax.scatter(np.degrees(t1_mean_rad), np.degrees(t2_mean_rad),
               marker='X', s=300, c=fg_colors_sc.get(vp, 'gray'),
               edgecolors='black', linewidths=2, zorder=10)

ax.set_xlabel(f'Angle 1: atan2({EV_Y_NAME}, {EV_X_NAME}) [degrees]', fontsize=12)
ax.set_ylabel('Angle 2: atan2(EV3, EV2) [degrees]', fontsize=12)
ax.set_title('2D angular viewpoint space — 8 viewpoints separate without supervision',
             fontsize=13)
ax.legend(fontsize=10, ncol=2, loc='lower right', markerscale=2)
ax.grid(True, alpha=0.3)
ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Final distance histogram: same-VP vs different-VP (3D Laplacian)
# ============================================================
# No island artifact — EV4 was dropped.

emb_hist_final = eigenvectors_L[:, 1:4]
d_final_sub = squareform(pdist(emb_hist_final[sub_h], 'euclidean'))[triu_h]

fig, ax = plt.subplots(figsize=(11, 6))
ax.hist(d_final_sub[same_relaxed],  bins=80, alpha=0.6, density=True,
        color='green', label=f'Same VP, relaxed (n={same_relaxed.sum()})')
ax.hist(d_final_sub[~same_relaxed], bins=80, alpha=0.6, density=True,
        color='red',   label=f'Diff VP (n={(~same_relaxed).sum()})')

auc_hist = roc_auc_score(same_relaxed, -d_final_sub)

ax.set_xlabel('Laplacian 3D distance', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'Distance distribution — Laplacian 3D (EVs 1-3)  |  AUC={auc_hist:.4f}',
             fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, d_final_sub.max() * 1.05)
plt.tight_layout()
plt.show()

print(f'Max distance: {d_final_sub.max():.4f}  (EV4 baseline had d_max = 0.3257 with a spurious island)')


In [ ]:
# ============================================================
# Interactive 3D: drag to rotate the EV1/EV2/EV3 space
# ============================================================
# Same as the earlier interactive cell — included in the summary
# so the full story lives at the end of the notebook.

import plotly.graph_objects as go

MAX_POINTS_SUMMARY = 6000
if len(valid_raw_vps) > MAX_POINTS_SUMMARY:
    rng_sum = np.random.RandomState(42)
    sub_sum = rng_sum.choice(len(valid_raw_vps), MAX_POINTS_SUMMARY, replace=False)
else:
    sub_sum = np.arange(len(valid_raw_vps))

fig_sum = go.Figure()
for vp in fg_sorted_sc:
    m = valid_raw_vps[sub_sum] == vp
    if m.sum() == 0:
        continue
    fig_sum.add_trace(go.Scatter3d(
        x=eigenvectors_L[sub_sum, 1][m],
        y=eigenvectors_L[sub_sum, 2][m],
        z=eigenvectors_L[sub_sum, 3][m],
        mode='markers',
        marker=dict(size=3, color=fg_colors_sc.get(vp, 'gray'), opacity=0.7),
        name=f'{vp} ({m.sum()})',
        hovertemplate=f'{vp}<br>EV1=%{{x:.4f}}<br>EV2=%{{y:.4f}}<br>EV3=%{{z:.4f}}<extra></extra>'
    ))

fig_sum.update_layout(
    title=f'Laplacian EVs 1-3 — interactive 3D (n={len(sub_sum)} subsample)',
    scene=dict(xaxis_title='EV1', yaxis_title='EV2', zaxis_title='EV3',
               aspectmode='cube'),
    width=950, height=750,
    legend=dict(itemsizing='constant'),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig_sum.show()


In [ ]:
# ============================================================
# Pair inspector (summary): sample detection pairs by distance
# ============================================================
# Adjust the three knobs at the top and re-run to see a different
# random pair from the specified distance range and relation.
#
# Uses the 3D Laplacian embedding (EVs 1-3).

# ---- knobs ----
DIST_MIN  = 0.01     # inclusive lower bound on 3D Laplacian distance
DIST_MAX  = 0.02     # inclusive upper bound
PAIR_TYPE = 'same'   # 'same' (relaxed same-VP) | 'diff' | 'any'
RANDOM    = True     # if False, take the first pair in the range
RNG_SEED  = None     # None for fresh draw each re-run
# ----------------

from PIL import Image as PILImage

assert emb_hist.shape[1] == 3, (
    f'emb_hist is {emb_hist.shape[1]}D — rerun the distance-histogram cell '
    f'after switching to EVs 1-3.'
)

det_to_annot_pi = {m.detection_id: m.gt_annotation for m in matched}

dist_mask = (dists_flat >= DIST_MIN) & (dists_flat <= DIST_MAX)
if PAIR_TYPE == 'same':
    pair_mask = dist_mask & same_relaxed
elif PAIR_TYPE == 'diff':
    pair_mask = dist_mask & ~same_relaxed
elif PAIR_TYPE == 'any':
    pair_mask = dist_mask
else:
    raise ValueError(f'PAIR_TYPE must be same/diff/any, got {PAIR_TYPE!r}')

n_pairs = pair_mask.sum()
print(f'Pairs with d in [{DIST_MIN}, {DIST_MAX}], type={PAIR_TYPE}: {n_pairs}')

if n_pairs == 0:
    print('No pairs found in this range — widen the bounds.')
else:
    pair_indices = np.where(pair_mask)[0]
    if RANDOM:
        rng_pi = np.random.RandomState(RNG_SEED)
        pick = rng_pi.choice(pair_indices)
    else:
        pick = pair_indices[0]

    i_sub, j_sub = triu_h[0][pick], triu_h[1][pick]
    i_global, j_global = sub_h[i_sub], sub_h[j_sub]
    dist_val = dists_flat[pick]

    def _angle1(gi):
        return np.degrees(np.arctan2(eigenvectors_L[gi, EV_Y], eigenvectors_L[gi, EV_X]))

    def _angle2(gi):
        return np.degrees(np.arctan2(eigenvectors_L[gi, 3], eigenvectors_L[gi, 2]))

    print(f'\n{"field":>14} {"Detection A":>30} {"Detection B":>30}')
    print('-' * 76)
    for label, fmt in [
        ('det_id',       lambda gi: valid_dets[gi][:28]),
        ('raw VP',       lambda gi: valid_raw_vps[gi]),
        ('coarse VP',    lambda gi: valid_vps[gi]),
        ('visibility',   lambda gi: f'{visibility[gi]:.3f}'),
        ('edge prox',    lambda gi: f'{edge_proximity[gi]:.4f}'),
        (f'angle1 ({EV_X_NAME},{EV_Y_NAME})',lambda gi: f'{_angle1(gi):+.1f}°'),
        ('angle2 (EV23)',lambda gi: f'{_angle2(gi):+.1f}°'),
        (EV_X_NAME,      lambda gi: f'{eigenvectors_L[gi, EV_X]:+.5f}'),
        (EV_Y_NAME,      lambda gi: f'{eigenvectors_L[gi, EV_Y]:+.5f}'),
        ('EV3',          lambda gi: f'{eigenvectors_L[gi, 3]:+.5f}'),
        ('identity',     lambda gi: str(det_to_annot_pi[valid_dets[gi]].individual_id)[:26]
                                    if valid_dets[gi] in det_to_annot_pi else '?'),
        ('image_uuid',   lambda gi: str(det_to_annot_pi[valid_dets[gi]].image_uuid)[:26]
                                    if valid_dets[gi] in det_to_annot_pi else '?'),
    ]:
        print(f'{label:>14} {fmt(i_global):>30} {fmt(j_global):>30}')
    print('-' * 76)
    ind_a = det_to_annot_pi[valid_dets[i_global]].individual_id if valid_dets[i_global] in det_to_annot_pi else None
    ind_b = det_to_annot_pi[valid_dets[j_global]].individual_id if valid_dets[j_global] in det_to_annot_pi else None
    same_identity = (ind_a is not None) and (ind_a == ind_b)

    print(f'{"3D distance":>14} {dist_val:>30.6f}')
    print(f'{"same-VP relaxed":>14} {str(same_relaxed[pick]):>30}')
    print(f'{"same identity":>14} {str(same_identity):>30}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 8))
    for ax, gi, label in [(axes[0], i_global, 'A'), (axes[1], j_global, 'B')]:
        did = valid_dets[gi]
        det_obj = dataset.get_detection(did)
        img = PILImage.open(det_obj.image_path).convert('RGB')
        img_w, img_h = img.size
        sx1, sy1, sx2, sy2 = det_obj.bbox.int().tolist()
        sx1, sy1 = max(0, sx1), max(0, sy1)
        sx2, sy2 = min(img_w, sx2), min(img_h, sy2)
        crop = img.crop((sx1, sy1, sx2, sy2))
        ax.imshow(crop)
        ind = det_to_annot_pi[did].individual_id if did in det_to_annot_pi else '?'
        ax.set_title(
            f'{label}: {valid_raw_vps[gi]} (coarse: {valid_vps[gi]})\n'
            f'vis={visibility[gi]:.2f}, edge={edge_proximity[gi]:.3f}\n'
            f'θ1={_angle1(gi):+.1f}°, θ2={_angle2(gi):+.1f}°\n'
            f'ind={str(ind)[:18]}',
            fontsize=11,
        )
        ax.axis('off')

    color = 'green' if same_relaxed[pick] else 'red'
    relation = 'SAME' if same_relaxed[pick] else 'DIFF'
    fig.suptitle(f'd={dist_val:.4f}, {relation} VP  (type={PAIR_TYPE})',
                 fontsize=14, y=1.02, color=color)
    plt.tight_layout()
    plt.show()
